# Enhanced MWPCA-KL Concept Drift Detection

**Improvements over the original script**

1. Multiple artificial drift injection strategies (gradual, sudden, incremental, recurring, feature-level)
2. Multiple drift detection methods (KL, MMD, Hotelling's T², PSI, energy distance)
3. **4-way drift-type classification** on the ResNet-KL branch — differentiates *concept*, *label* (prior), *feature* (covariate), and general *data* drift
4. Consecutive-window (sliding) KL for detecting *when* drift stabilises
5. Symmetric KL (Jensen-Shannon divergence) for more robust comparison
6. Proper bootstrap-based thresholds as alternative to chi² assumption
7. Comprehensive visualisation dashboard
8. **Ground-truth drift timeline** recorded during stream generation and overlaid on every detection plot (background shading shows the true OOC-fraction and feature-augmentation intensity per window)
9. **Detection-latency metrics** per method/per experiment — reports both `Δfirst` (latency vs first true drift window) and `Δsubst` (latency vs window where drift exceeds 20%)

**Drift types the ResNet-KL branch distinguishes**

| Category | P(X) shifted | P(Y) shifted | P(Y\|X) shifted | Injected by | 4-way classifier signals |
|----------|:---:|:---:|:---:|---|---|
| `label`    | no  | **yes** | no | class-mix shift (sudden / gradual / incremental / recurring) | prior-shift only |
| `feature`  | **yes** | no | no | image augmentations (noise, blur, saturation, …) | covariate-shift only |
| `concept`  | yes | yes | **yes** | concurrent class-mix + augmentation (not injected in the default config) | prior-shift AND covariate-shift |
| `none`     | no  | no | no | baseline / stable periods | neither |

CIFAR-10 with fixed labels cannot express true P(Y\|X) drift directly, so this
notebook treats *concurrent* prior + covariate shift as the empirical signature
of concept drift. To see the `concept` label fire, uncomment the
`Concept-like (label + feature)` entry in the `drift_configs` cell below.

A dedicated `DriftType.CONCEPT` stream creator (label-flipping on in-control
images) is also available if you want a stream where P(X) stays constant and
only P(Y\|X) changes — see `create_stream_concept` in the drift-stream cell.

**New plots**
- **drift_dashboard_truth_\*** — 6-panel ResNet dashboard, each subplot shaded with the true drift timeline; dotted red line marks first detection.
- **all_methods_truth_\*** — 3×3 side-by-side of every framework (ResNet-KL/JSD/T², VAE recon/latent-KL, ADWIN×2, DDM, EDDM), each annotated with `Δfirst` / `Δsubst`.
- **classification_timeline_\*** — per-window 4-way classification as a scatter timeline, compared to ground truth.
- **latency_\*** — horizontal bar chart of first-detection latency per method, for both 'first drift' and 'substantial drift' references.
- **latency_heatmap_cross_experiment** — cross-experiment latency matrix (methods × experiments).


In [1]:
# -*- coding: utf-8 -*-
%matplotlib inline

import os
import io
import random
import warnings
import gc
from enum import Enum
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Callable

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.decomposition import PCA
from scipy.stats import chi2, ks_2samp
from scipy.spatial.distance import cdist

warnings.filterwarnings('ignore')

/Users/theo/Documents/Thesis Scripts/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(42)

if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using MPS (Metal Performance Shaders) device")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("Using CUDA device")
else:
    device = torch.device('cpu')
    print("Using CPU device")


Using MPS (Metal Performance Shaders) device


In [ ]:
# =============================================================================
# NEW CELL A: Extend DriftType and DriftConfig with a CONCEPT drift scenario
# =============================================================================
# Drop-in replacement for the existing DriftType / DriftConfig cell.

from enum import Enum
from dataclasses import dataclass, field
from typing import List, Optional

class DriftType(Enum):
    """Types of drift that can be injected.

    ── Categorical distinctions ──────────────────────────────────────────────
      LABEL / PRIOR drift : P(Y) changes; P(X|Y) stable
      FEATURE / COVARIATE : P(X) changes; P(Y|X) stable
      CONCEPT             : P(Y|X) changes; driven by relabeling / class remap
      DATA                : general P(X,Y) change (used for classifier output)
    ── Temporal patterns (how the shift is introduced) ──────────────────────
      SUDDEN, GRADUAL, INCREMENTAL, RECURRING — these modulate *when*
      OOC / feature / concept shift appears across the stream.
    """
    SUDDEN             = "sudden"
    GRADUAL            = "gradual"
    INCREMENTAL        = "incremental"
    RECURRING          = "recurring"
    FEATURE_COVARIATE  = "feature"
    CONCEPT            = "concept"      # NEW — label-flip / class-remap drift


@dataclass
class DriftConfig:
    """Configuration for a drift injection experiment."""
    drift_type: DriftType
    in_control_classes: List[int]
    out_of_control_classes: List[int]
    n_in_control_start: int = 2000
    n_transition: int = 8000
    n_out_of_control_end: int = 10000
    # Sudden drift
    changepoint_frac: float = 0.3
    # Recurring drift
    cycle_length: int = 2000
    n_cycles: int = 3
    # Feature / covariate drift
    noise_std: float = 0.5
    brightness_shift: float = 0.4
    saturation_factor: float = 1.8
    contrast_factor: float = 0.5
    blur_kernel_size: int = 5
    blur_sigma: float = 2.0
    jpeg_quality: int = 10
    elastic_alpha: float = 50.0
    elastic_sigma: float = 5.0
    color_jitter_strength: float = 0.5
    augmentation_types: Optional[List[str]] = None

    # ── NEW: Concept drift parameters ─────────────────────────────────────
    # For concept drift, we deliberately change P(Y|X) by either:
    #   (a) re-labelling a subset of the in-control class to a different label
    #       while the images remain visually identical, OR
    #   (b) swapping the semantic meaning of two class IDs mid-stream.
    # This is "true" concept drift: the relationship between X and Y changes
    # while the marginal P(X) can stay stable or also shift.
    concept_flip_classes: Optional[List[int]] = None   # which labels to flip
    concept_flip_target: Optional[int] = None          # new label value
    concept_flip_rate: float = 1.0                     # fraction of flipped samples


In [4]:
def load_cifar10(data_dir: str = './data'):
    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    os.makedirs(data_dir, exist_ok=True)
    trainset = torchvision.datasets.CIFAR10(root=data_dir, train=True,  download=True, transform=transform)
    testset  = torchvision.datasets.CIFAR10(root=data_dir, train=False, download=True, transform=transform)
    return trainset, testset


def load_all_data(trainset, testset):
    """Load all images and labels into memory."""
    all_data, all_labels = [], []
    for ds, name in [(trainset, "train"), (testset, "test")]:
        print(f"Loading {name} set ({len(ds)} samples)...")
        for i in range(len(ds)):
            img, label = ds[i]
            all_data.append(img)
            all_labels.append(label)
    print(f"Total samples loaded: {len(all_data)}")
    return all_data, all_labels


In [5]:
print("Loading CIFAR-10...")
trainset, testset = load_cifar10()
all_data, all_labels = load_all_data(trainset, testset)


Loading CIFAR-10...
Loading train set (50000 samples)...
Loading test set (10000 samples)...
Total samples loaded: 60000


In [6]:
# ── ImageNet normalization helpers ─────────────────────────────────────────────
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denormalize_imagenet(x: torch.Tensor) -> torch.Tensor:
    """Undo ImageNet normalization: (C,H,W) or (B,C,H,W) → [0, 1]."""
    mean = IMAGENET_MEAN.to(x.device)
    std  = IMAGENET_STD.to(x.device)
    if x.dim() == 4:
        mean = mean.unsqueeze(0)
        std  = std.unsqueeze(0)
    return (x * std + mean).clamp(0.0, 1.0)

def renormalize_imagenet(x: torch.Tensor) -> torch.Tensor:
    """Re-apply ImageNet normalization: [0,1] → normalized."""
    mean = IMAGENET_MEAN.to(x.device)
    std  = IMAGENET_STD.to(x.device)
    if x.dim() == 4:
        mean = mean.unsqueeze(0)
        std  = std.unsqueeze(0)
    return (x - mean) / std


# ── Individual augmentation functions ─────────────────────────────────────────
# Each takes an ImageNet-normalized (C,H,W) tensor, a DriftConfig, and a
# progress float in [0,1] controlling augmentation intensity.

def apply_feature_perturbation(img_tensor: torch.Tensor,
                                noise_std: float = 0.5,
                                brightness_shift: float = 0.4) -> torch.Tensor:
    """Apply pixel-level perturbation to simulate covariate / feature drift."""
    perturbed = img_tensor.clone()
    perturbed += torch.randn_like(perturbed) * noise_std
    perturbed += brightness_shift
    return perturbed


def apply_noise(img: torch.Tensor, config: DriftConfig, progress: float) -> torch.Tensor:
    """Additive Gaussian noise + brightness shift (original perturbation)."""
    return apply_feature_perturbation(
        img, config.noise_std * progress, config.brightness_shift * progress)


def apply_saturation_shift(img: torch.Tensor, config: DriftConfig, progress: float) -> torch.Tensor:
    """Shift saturation by converting to HSV, scaling S, converting back."""
    raw = denormalize_imagenet(img)  # (C,H,W) in [0,1]
    r, g, b = raw[0], raw[1], raw[2]
    maxc = torch.max(raw, dim=0).values
    minc = torch.min(raw, dim=0).values
    delta = maxc - minc + 1e-8

    # Saturation
    s = delta / (maxc + 1e-8)
    factor = 1.0 + (config.saturation_factor - 1.0) * progress
    s_new = (s * factor).clamp(0.0, 1.0)

    # Hue (in [0,6))
    h = torch.zeros_like(r)
    mask_r = (maxc == r) & (delta > 1e-8)
    mask_g = (maxc == g) & (delta > 1e-8) & ~mask_r
    mask_b = (delta > 1e-8) & ~mask_r & ~mask_g
    h[mask_r] = ((g[mask_r] - b[mask_r]) / delta[mask_r]) % 6.0
    h[mask_g] = ((b[mask_g] - r[mask_g]) / delta[mask_g]) + 2.0
    h[mask_b] = ((r[mask_b] - g[mask_b]) / delta[mask_b]) + 4.0

    # HSV to RGB with new saturation
    v = maxc
    c_val = v * s_new
    x_val = c_val * (1 - torch.abs((h % 2.0) - 1.0))
    m_val = v - c_val

    hi = h.long() % 6
    r_out = torch.zeros_like(r)
    g_out = torch.zeros_like(r)
    b_out = torch.zeros_like(r)

    # Sector 0: R=C, G=X, B=0
    m = (hi == 0)
    if m.any():
        r_out[m] = c_val[m] + m_val[m]; g_out[m] = x_val[m] + m_val[m]; b_out[m] = m_val[m]
    # Sector 1: R=X, G=C, B=0
    m = (hi == 1)
    if m.any():
        r_out[m] = x_val[m] + m_val[m]; g_out[m] = c_val[m] + m_val[m]; b_out[m] = m_val[m]
    # Sector 2: R=0, G=C, B=X
    m = (hi == 2)
    if m.any():
        r_out[m] = m_val[m]; g_out[m] = c_val[m] + m_val[m]; b_out[m] = x_val[m] + m_val[m]
    # Sector 3: R=0, G=X, B=C
    m = (hi == 3)
    if m.any():
        r_out[m] = m_val[m]; g_out[m] = x_val[m] + m_val[m]; b_out[m] = c_val[m] + m_val[m]
    # Sector 4: R=X, G=0, B=C
    m = (hi == 4)
    if m.any():
        r_out[m] = x_val[m] + m_val[m]; g_out[m] = m_val[m]; b_out[m] = c_val[m] + m_val[m]
    # Sector 5: R=C, G=0, B=X
    m = (hi == 5)
    if m.any():
        r_out[m] = c_val[m] + m_val[m]; g_out[m] = m_val[m]; b_out[m] = x_val[m] + m_val[m]

    result = torch.stack([r_out, g_out, b_out], dim=0).clamp(0.0, 1.0)
    return renormalize_imagenet(result)


def apply_contrast_change(img: torch.Tensor, config: DriftConfig, progress: float) -> torch.Tensor:
    """Scale pixel deviation from per-channel mean."""
    raw = denormalize_imagenet(img)
    factor = 1.0 + (config.contrast_factor - 1.0) * progress  # e.g. 1.0 → 0.5
    mean = raw.mean(dim=[1, 2], keepdim=True)
    result = mean + factor * (raw - mean)
    return renormalize_imagenet(result.clamp(0.0, 1.0))


def _make_gaussian_kernel(kernel_size: int, sigma: float) -> torch.Tensor:
    """Create a 2D Gaussian kernel of shape (1, 1, k, k)."""
    ax = torch.arange(kernel_size, dtype=torch.float32) - (kernel_size - 1) / 2.0
    gauss = torch.exp(-0.5 * (ax / sigma) ** 2)
    kernel = gauss.outer(gauss)
    kernel = kernel / kernel.sum()
    return kernel.view(1, 1, kernel_size, kernel_size)


def apply_gaussian_blur(img: torch.Tensor, config: DriftConfig, progress: float) -> torch.Tensor:
    """Apply Gaussian blur with intensity scaled by progress."""
    sigma = config.blur_sigma * progress
    if sigma < 0.1:
        return img.clone()
    ks = config.blur_kernel_size
    pad = ks // 2
    kernel = _make_gaussian_kernel(ks, sigma).to(img.device)
    kernel = kernel.repeat(3, 1, 1, 1)  # (3, 1, k, k) for groups=3
    x = img.unsqueeze(0)  # (1, C, H, W)
    blurred = F.conv2d(x, kernel, padding=pad, groups=3).squeeze(0)
    return blurred


def apply_jpeg_artifacts(img: torch.Tensor, config: DriftConfig, progress: float) -> torch.Tensor:
    """JPEG compression artifacts via PIL round-trip."""
    quality = int(100 - (100 - config.jpeg_quality) * progress)
    quality = max(1, min(100, quality))
    if quality >= 95:
        return img.clone()
    raw = denormalize_imagenet(img)  # (C,H,W) [0,1]
    np_img = (raw.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    pil_img = Image.fromarray(np_img)
    buf = io.BytesIO()
    pil_img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    pil_compressed = Image.open(buf)
    np_compressed = np.array(pil_compressed).astype(np.float32) / 255.0
    result = torch.from_numpy(np_compressed).permute(2, 0, 1).to(img.device)
    return renormalize_imagenet(result)


def apply_elastic_distortion(img: torch.Tensor, config: DriftConfig, progress: float) -> torch.Tensor:
    """Elastic distortion using random displacement fields + grid_sample."""
    alpha = config.elastic_alpha * progress
    if alpha < 1.0:
        return img.clone()
    sigma = config.elastic_sigma
    _, h, w = img.shape

    # Random displacement fields smoothed with Gaussian
    ks = int(6 * sigma + 1)
    if ks % 2 == 0:
        ks += 1
    pad = ks // 2
    gk = _make_gaussian_kernel(ks, sigma).to(img.device)

    dx = torch.randn(1, 1, h, w, device=img.device)
    dy = torch.randn(1, 1, h, w, device=img.device)
    dx = F.conv2d(dx, gk, padding=pad) * alpha
    dy = F.conv2d(dy, gk, padding=pad) * alpha

    # Normalize displacements to [-1, 1] grid space
    dx = dx.squeeze() / (w / 2.0)
    dy = dy.squeeze() / (h / 2.0)

    # Build sampling grid
    grid_y, grid_x = torch.meshgrid(
        torch.linspace(-1, 1, h, device=img.device),
        torch.linspace(-1, 1, w, device=img.device),
        indexing='ij')
    grid = torch.stack([grid_x + dx, grid_y + dy], dim=-1).unsqueeze(0)

    x = img.unsqueeze(0)
    warped = F.grid_sample(x, grid, mode='bilinear', padding_mode='reflection',
                           align_corners=True).squeeze(0)
    return warped


def apply_color_jitter(img: torch.Tensor, config: DriftConfig, progress: float) -> torch.Tensor:
    """Per-channel multiplicative scaling to simulate color drift."""
    strength = config.color_jitter_strength * progress
    if strength < 0.01:
        return img.clone()
    raw = denormalize_imagenet(img)
    # Random per-channel factor in [1 - strength, 1 + strength]
    factors = 1.0 + (torch.rand(3, 1, 1, device=img.device) * 2 - 1) * strength
    result = (raw * factors).clamp(0.0, 1.0)
    return renormalize_imagenet(result)


# ── Augmentation suite dispatcher ─────────────────────────────────────────────

AUGMENTATION_REGISTRY = {
    'noise':        apply_noise,
    'brightness':   lambda img, cfg, p: apply_feature_perturbation(
                        img, 0, cfg.brightness_shift * p),
    'saturation':   apply_saturation_shift,
    'contrast':     apply_contrast_change,
    'blur':         apply_gaussian_blur,
    'jpeg':         apply_jpeg_artifacts,
    'elastic':      apply_elastic_distortion,
    'color_jitter': apply_color_jitter,
}


def apply_augmentation_suite(img: torch.Tensor, config: DriftConfig,
                             progress: float) -> torch.Tensor:
    """Apply selected augmentations to an image based on config.

    If config.augmentation_types is None, all augmentations are applied.
    Otherwise, only the listed augmentation names are applied.
    """
    types = config.augmentation_types
    if types is None:
        types = list(AUGMENTATION_REGISTRY.keys())

    result = img.clone()
    for aug_name in types:
        if aug_name in AUGMENTATION_REGISTRY:
            result = AUGMENTATION_REGISTRY[aug_name](result, config, progress)
    return result


# ── Drift stream creators ────────────────────────────────────────────────────

def create_stream_sudden(all_data, all_labels, config: DriftConfig):
    """SUDDEN DRIFT: clean in-control data, then instant switch at changepoint."""
    in_idx  = [i for i, l in enumerate(all_labels) if l in config.in_control_classes]
    out_idx = [i for i, l in enumerate(all_labels) if l in config.out_of_control_classes]
    random.shuffle(in_idx); random.shuffle(out_idx)

    total       = config.n_in_control_start + config.n_transition + config.n_out_of_control_end
    changepoint = int(total * config.changepoint_frac)

    stream_data, stream_labels = [], []
    in_ptr, out_ptr = 0, 0

    for i in range(total):
        if i < changepoint:
            if in_ptr < len(in_idx):
                idx = in_idx[in_ptr]; in_ptr += 1
                stream_data.append(all_data[idx]); stream_labels.append(all_labels[idx])
        else:
            if out_ptr < len(out_idx):
                idx = out_idx[out_ptr]; out_ptr += 1
                stream_data.append(all_data[idx]); stream_labels.append(all_labels[idx])

    print(f"[Sudden] Stream: {len(stream_data)} samples, changepoint at index {changepoint}")
    return stream_data, stream_labels


def create_stream_gradual(all_data, all_labels, config: DriftConfig):
    """GRADUAL DRIFT: linearly increasing probability of OOC samples."""
    in_idx  = [i for i, l in enumerate(all_labels) if l in config.in_control_classes]
    out_idx = [i for i, l in enumerate(all_labels) if l in config.out_of_control_classes]
    random.shuffle(in_idx); random.shuffle(out_idx)

    stream_indices = list(in_idx[:config.n_in_control_start])
    remaining_in   = in_idx[config.n_in_control_start:]
    remaining_out  = list(out_idx)

    for i in range(config.n_transition):
        p_out = i / config.n_transition
        if random.random() < p_out and remaining_out:
            stream_indices.append(remaining_out.pop(0))
        elif remaining_in:
            stream_indices.append(remaining_in.pop(0))
        elif remaining_out:
            stream_indices.append(remaining_out.pop(0))

    stream_indices.extend(remaining_out[:config.n_out_of_control_end])
    stream_data   = [all_data[i]   for i in stream_indices]
    stream_labels = [all_labels[i] for i in stream_indices]
    print(f"[Gradual] Stream: {len(stream_data)} samples")
    return stream_data, stream_labels


def create_stream_incremental(all_data, all_labels, config: DriftConfig):
    """INCREMENTAL DRIFT: OOC classes introduced one at a time in stages."""
    in_idx = [i for i, l in enumerate(all_labels) if l in config.in_control_classes]
    random.shuffle(in_idx)

    ooc_by_class = {}
    for cls in config.out_of_control_classes:
        ooc_by_class[cls] = [i for i, l in enumerate(all_labels) if l == cls]
        random.shuffle(ooc_by_class[cls])

    n_stages  = len(config.out_of_control_classes)
    stage_len = config.n_transition // n_stages

    stream_data, stream_labels = [], []
    for i in range(min(config.n_in_control_start, len(in_idx))):
        stream_data.append(all_data[in_idx[i]])
        stream_labels.append(all_labels[in_idx[i]])

    in_ptr   = config.n_in_control_start
    ooc_ptrs = {cls: 0 for cls in config.out_of_control_classes}
    active_ooc_classes = []

    for stage in range(n_stages):
        active_ooc_classes.append(config.out_of_control_classes[stage])
        for _ in range(stage_len):
            p_out = (stage + 1) / (n_stages + 1)
            if random.random() < p_out and active_ooc_classes:
                cls      = random.choice(active_ooc_classes)
                idx_list = ooc_by_class[cls]
                ptr      = ooc_ptrs[cls]
                if ptr < len(idx_list):
                    stream_data.append(all_data[idx_list[ptr]])
                    stream_labels.append(all_labels[idx_list[ptr]])
                    ooc_ptrs[cls] = ptr + 1
                    continue
            if in_ptr < len(in_idx):
                stream_data.append(all_data[in_idx[in_ptr]])
                stream_labels.append(all_labels[in_idx[in_ptr]])
                in_ptr += 1

    all_remaining_ooc = []
    for cls in config.out_of_control_classes:
        all_remaining_ooc.extend(ooc_by_class[cls][ooc_ptrs[cls]:])
    random.shuffle(all_remaining_ooc)
    for idx in all_remaining_ooc[:config.n_out_of_control_end]:
        stream_data.append(all_data[idx])
        stream_labels.append(all_labels[idx])

    print(f"[Incremental] Stream: {len(stream_data)} samples, {n_stages} stages")
    return stream_data, stream_labels


def create_stream_recurring(all_data, all_labels, config: DriftConfig):
    """RECURRING DRIFT: alternating periods of in-control and out-of-control."""
    in_idx  = [i for i, l in enumerate(all_labels) if l in config.in_control_classes]
    out_idx = [i for i, l in enumerate(all_labels) if l in config.out_of_control_classes]
    random.shuffle(in_idx); random.shuffle(out_idx)

    stream_data, stream_labels = [], []
    in_ptr, out_ptr = 0, 0
    half_cycle = config.cycle_length // 2

    for i in range(config.n_in_control_start):
        if in_ptr < len(in_idx):
            stream_data.append(all_data[in_idx[in_ptr]])
            stream_labels.append(all_labels[in_idx[in_ptr]])
            in_ptr += 1

    for _ in range(config.n_cycles):
        for _ in range(half_cycle):
            if out_ptr < len(out_idx):
                stream_data.append(all_data[out_idx[out_ptr]])
                stream_labels.append(all_labels[out_idx[out_ptr]])
                out_ptr += 1
        for _ in range(half_cycle):
            if in_ptr < len(in_idx):
                stream_data.append(all_data[in_idx[in_ptr]])
                stream_labels.append(all_labels[in_idx[in_ptr]])
                in_ptr += 1

    print(f"[Recurring] Stream: {len(stream_data)} samples, {config.n_cycles} cycles")
    return stream_data, stream_labels


def create_stream_feature_drift(all_data, all_labels, config: DriftConfig):
    """FEATURE / COVARIATE DRIFT: same labels, augmentation intensity ramps up."""
    in_idx = [i for i, l in enumerate(all_labels) if l in config.in_control_classes]
    random.shuffle(in_idx)

    total = config.n_in_control_start + config.n_transition + config.n_out_of_control_end
    stream_data, stream_labels = [], []

    for i in range(min(total, len(in_idx))):
        img   = all_data[in_idx[i % len(in_idx)]]
        label = all_labels[in_idx[i % len(in_idx)]]
        if i < config.n_in_control_start:
            stream_data.append(img)
        elif i < config.n_in_control_start + config.n_transition:
            progress = (i - config.n_in_control_start) / config.n_transition
            stream_data.append(apply_augmentation_suite(img, config, progress))
        else:
            stream_data.append(apply_augmentation_suite(img, config, 1.0))
        stream_labels.append(label)

    aug_desc = config.augmentation_types or "all"
    print(f"[Feature Drift] Stream: {len(stream_data)} samples "
          f"(augmentations: {aug_desc})")
    return stream_data, stream_labels


DRIFT_CREATORS = {
    DriftType.SUDDEN:           create_stream_sudden,
    DriftType.GRADUAL:          create_stream_gradual,
    DriftType.INCREMENTAL:      create_stream_incremental,
    DriftType.RECURRING:        create_stream_recurring,
    DriftType.FEATURE_COVARIATE: create_stream_feature_drift,
}


def create_drift_stream(all_data, all_labels, config: DriftConfig):
    return DRIFT_CREATORS[config.drift_type](all_data, all_labels, config)

In [ ]:

# =============================================================================
# NEW CELL B: Ground-truth drift timeline + concept drift stream creator
# =============================================================================
# Insert this cell AFTER the existing drift-stream-creators cell (Cell 6)
# and BEFORE create_windows (Cell 7).
#
# Every stream creator now also returns a `drift_truth` dict describing the
# true drift timeline so we can overlay it on detection plots and measure
# detection latency.

import numpy as np
import random
import copy


# ──────────────────────────────────────────────────────────────────────────────
# Ground-truth timeline structure
# ──────────────────────────────────────────────────────────────────────────────
# drift_truth = {
#   'stream_length':      int,
#   'in_control_mask':    np.ndarray of bool, True where sample is in-control
#   'feature_intensity':  np.ndarray of float in [0,1], augmentation strength
#   'concept_mask':       np.ndarray of bool, True where label has been flipped
#   'changepoints':       list of (sample_idx, drift_kind) tuples
#                         drift_kind ∈ {'label', 'feature', 'concept'}
#   'true_drift_type':    one of {'label', 'feature', 'concept', 'mixed'}
#                         — the ground-truth *kind* of drift injected
# }
# ──────────────────────────────────────────────────────────────────────────────

def _empty_truth(n: int, true_type: str = 'none'):
    return {
        'stream_length':     n,
        'in_control_mask':   np.ones(n, dtype=bool),
        'feature_intensity': np.zeros(n, dtype=float),
        'concept_mask':      np.zeros(n, dtype=bool),
        'changepoints':      [],
        'true_drift_type':   true_type,
    }


# ──────────────────────────────────────────────────────────────────────────────
# Re-implementations of each stream creator that also build drift_truth
# ──────────────────────────────────────────────────────────────────────────────

def create_stream_sudden(all_data, all_labels, config: DriftConfig):
    in_idx  = [i for i, l in enumerate(all_labels) if l in config.in_control_classes]
    out_idx = [i for i, l in enumerate(all_labels) if l in config.out_of_control_classes]
    random.shuffle(in_idx); random.shuffle(out_idx)

    total       = config.n_in_control_start + config.n_transition + config.n_out_of_control_end
    changepoint = int(total * config.changepoint_frac)

    stream_data, stream_labels = [], []
    truth = _empty_truth(total, true_type='label')
    in_ptr, out_ptr = 0, 0

    for i in range(total):
        if i < changepoint:
            if in_ptr < len(in_idx):
                idx = in_idx[in_ptr]; in_ptr += 1
                stream_data.append(all_data[idx]); stream_labels.append(all_labels[idx])
                truth['in_control_mask'][i] = True
        else:
            if out_ptr < len(out_idx):
                idx = out_idx[out_ptr]; out_ptr += 1
                stream_data.append(all_data[idx]); stream_labels.append(all_labels[idx])
                truth['in_control_mask'][i] = False

    truth['stream_length'] = len(stream_data)
    truth['in_control_mask'] = truth['in_control_mask'][:len(stream_data)]
    truth['feature_intensity'] = truth['feature_intensity'][:len(stream_data)]
    truth['concept_mask'] = truth['concept_mask'][:len(stream_data)]
    truth['changepoints'].append((changepoint, 'label'))
    print(f"[Sudden] Stream: {len(stream_data)} samples, changepoint at index {changepoint}")
    return stream_data, stream_labels, truth


def create_stream_gradual(all_data, all_labels, config: DriftConfig):
    in_idx  = [i for i, l in enumerate(all_labels) if l in config.in_control_classes]
    out_idx = [i for i, l in enumerate(all_labels) if l in config.out_of_control_classes]
    random.shuffle(in_idx); random.shuffle(out_idx)

    stream_indices = list(in_idx[:config.n_in_control_start])
    is_in_control  = [True] * len(stream_indices)
    remaining_in   = in_idx[config.n_in_control_start:]
    remaining_out  = list(out_idx)

    transition_start = len(stream_indices)

    for i in range(config.n_transition):
        p_out = i / config.n_transition
        if random.random() < p_out and remaining_out:
            stream_indices.append(remaining_out.pop(0))
            is_in_control.append(False)
        elif remaining_in:
            stream_indices.append(remaining_in.pop(0))
            is_in_control.append(True)
        elif remaining_out:
            stream_indices.append(remaining_out.pop(0))
            is_in_control.append(False)

    transition_end = len(stream_indices)
    # Post-transition: pure OOC
    stream_indices.extend(remaining_out[:config.n_out_of_control_end])
    is_in_control.extend([False] * min(config.n_out_of_control_end, len(remaining_out)))

    stream_data   = [all_data[i]   for i in stream_indices]
    stream_labels = [all_labels[i] for i in stream_indices]

    n = len(stream_data)
    truth = _empty_truth(n, true_type='label')
    truth['in_control_mask'] = np.array(is_in_control[:n], dtype=bool)
    truth['changepoints'].append((transition_start, 'label'))
    if transition_end < n:
        truth['changepoints'].append((transition_end, 'label'))
    print(f"[Gradual] Stream: {n} samples")
    return stream_data, stream_labels, truth


def create_stream_incremental(all_data, all_labels, config: DriftConfig):
    in_idx = [i for i, l in enumerate(all_labels) if l in config.in_control_classes]
    random.shuffle(in_idx)

    ooc_by_class = {}
    for cls in config.out_of_control_classes:
        ooc_by_class[cls] = [i for i, l in enumerate(all_labels) if l == cls]
        random.shuffle(ooc_by_class[cls])

    n_stages  = len(config.out_of_control_classes)
    stage_len = config.n_transition // n_stages

    stream_data, stream_labels = [], []
    is_in_control = []
    for i in range(min(config.n_in_control_start, len(in_idx))):
        stream_data.append(all_data[in_idx[i]])
        stream_labels.append(all_labels[in_idx[i]])
        is_in_control.append(True)

    in_ptr   = config.n_in_control_start
    ooc_ptrs = {cls: 0 for cls in config.out_of_control_classes}
    active_ooc_classes = []

    stage_changepoints = []
    for stage in range(n_stages):
        stage_changepoints.append((len(stream_data), 'label'))
        active_ooc_classes.append(config.out_of_control_classes[stage])
        for _ in range(stage_len):
            p_out = (stage + 1) / (n_stages + 1)
            if random.random() < p_out and active_ooc_classes:
                cls      = random.choice(active_ooc_classes)
                idx_list = ooc_by_class[cls]
                ptr      = ooc_ptrs[cls]
                if ptr < len(idx_list):
                    stream_data.append(all_data[idx_list[ptr]])
                    stream_labels.append(all_labels[idx_list[ptr]])
                    is_in_control.append(False)
                    ooc_ptrs[cls] = ptr + 1
                    continue
            if in_ptr < len(in_idx):
                stream_data.append(all_data[in_idx[in_ptr]])
                stream_labels.append(all_labels[in_idx[in_ptr]])
                is_in_control.append(True)
                in_ptr += 1

    all_remaining_ooc = []
    for cls in config.out_of_control_classes:
        all_remaining_ooc.extend(ooc_by_class[cls][ooc_ptrs[cls]:])
    random.shuffle(all_remaining_ooc)
    for idx in all_remaining_ooc[:config.n_out_of_control_end]:
        stream_data.append(all_data[idx])
        stream_labels.append(all_labels[idx])
        is_in_control.append(False)

    n = len(stream_data)
    truth = _empty_truth(n, true_type='label')
    truth['in_control_mask'] = np.array(is_in_control[:n], dtype=bool)
    truth['changepoints']    = stage_changepoints
    print(f"[Incremental] Stream: {n} samples, {n_stages} stages")
    return stream_data, stream_labels, truth


def create_stream_recurring(all_data, all_labels, config: DriftConfig):
    in_idx  = [i for i, l in enumerate(all_labels) if l in config.in_control_classes]
    out_idx = [i for i, l in enumerate(all_labels) if l in config.out_of_control_classes]
    random.shuffle(in_idx); random.shuffle(out_idx)

    stream_data, stream_labels = [], []
    is_in_control = []
    in_ptr, out_ptr = 0, 0
    half_cycle = config.cycle_length // 2
    changepoints = []

    for i in range(config.n_in_control_start):
        if in_ptr < len(in_idx):
            stream_data.append(all_data[in_idx[in_ptr]])
            stream_labels.append(all_labels[in_idx[in_ptr]])
            is_in_control.append(True)
            in_ptr += 1

    for cycle_num in range(config.n_cycles):
        # OOC half
        changepoints.append((len(stream_data), 'label'))
        for _ in range(half_cycle):
            if out_ptr < len(out_idx):
                stream_data.append(all_data[out_idx[out_ptr]])
                stream_labels.append(all_labels[out_idx[out_ptr]])
                is_in_control.append(False)
                out_ptr += 1
        # In-control half
        changepoints.append((len(stream_data), 'label'))
        for _ in range(half_cycle):
            if in_ptr < len(in_idx):
                stream_data.append(all_data[in_idx[in_ptr]])
                stream_labels.append(all_labels[in_idx[in_ptr]])
                is_in_control.append(True)
                in_ptr += 1

    n = len(stream_data)
    truth = _empty_truth(n, true_type='label')
    truth['in_control_mask'] = np.array(is_in_control[:n], dtype=bool)
    truth['changepoints']    = changepoints
    print(f"[Recurring] Stream: {n} samples, {config.n_cycles} cycles")
    return stream_data, stream_labels, truth


def create_stream_feature_drift(all_data, all_labels, config: DriftConfig):
    in_idx = [i for i, l in enumerate(all_labels) if l in config.in_control_classes]
    random.shuffle(in_idx)

    total = config.n_in_control_start + config.n_transition + config.n_out_of_control_end
    stream_data, stream_labels = [], []
    feature_intensity = []

    for i in range(min(total, len(in_idx))):
        img   = all_data[in_idx[i % len(in_idx)]]
        label = all_labels[in_idx[i % len(in_idx)]]
        if i < config.n_in_control_start:
            stream_data.append(img)
            feature_intensity.append(0.0)
        elif i < config.n_in_control_start + config.n_transition:
            progress = (i - config.n_in_control_start) / config.n_transition
            stream_data.append(apply_augmentation_suite(img, config, progress))
            feature_intensity.append(progress)
        else:
            stream_data.append(apply_augmentation_suite(img, config, 1.0))
            feature_intensity.append(1.0)
        stream_labels.append(label)

    n = len(stream_data)
    truth = _empty_truth(n, true_type='feature')
    truth['feature_intensity'] = np.array(feature_intensity[:n], dtype=float)
    # Labels are stable (all in-control), so in_control_mask stays True
    truth['changepoints'].append((config.n_in_control_start, 'feature'))
    if config.n_in_control_start + config.n_transition < n:
        truth['changepoints'].append((config.n_in_control_start + config.n_transition, 'feature'))
    aug_desc = config.augmentation_types or "all"
    print(f"[Feature Drift] Stream: {n} samples (augmentations: {aug_desc})")
    return stream_data, stream_labels, truth


def create_stream_concept(all_data, all_labels, config: DriftConfig):
    """CONCEPT DRIFT: P(Y|X) changes. Images stay in-control visually, but
    starting at the changepoint, a fraction of their labels are remapped to a
    different target class — the same X now maps to a different Y.

    This is the cleanest simulation of concept drift: feature marginal P(X) is
    preserved, only the conditional P(Y|X) shifts.
    """
    # Default concept flip: take in-control samples and remap their labels to
    # an OOC class ID. If not specified, flip IC→first OOC class.
    flip_classes = config.concept_flip_classes
    if flip_classes is None:
        flip_classes = config.in_control_classes
    flip_target = config.concept_flip_target
    if flip_target is None:
        flip_target = config.out_of_control_classes[0]

    # Only use in-control images so the feature distribution stays constant.
    in_idx = [i for i, l in enumerate(all_labels) if l in config.in_control_classes]
    random.shuffle(in_idx)

    total = config.n_in_control_start + config.n_transition + config.n_out_of_control_end
    changepoint = config.n_in_control_start
    end_of_transition = config.n_in_control_start + config.n_transition

    stream_data, stream_labels = [], []
    concept_mask = []

    for i in range(min(total, len(in_idx))):
        idx = in_idx[i % len(in_idx)]
        img   = all_data[idx]
        label = all_labels[idx]

        # Decide whether to flip this sample's label
        if i < changepoint:
            # Pre-drift: original label
            flipped = False
        elif i < end_of_transition:
            # Gradual ramp of flip probability over the transition region
            progress = (i - changepoint) / config.n_transition
            p_flip = progress * config.concept_flip_rate
            flipped = (label in flip_classes) and (random.random() < p_flip)
        else:
            # Post-drift: flip at the configured rate
            flipped = (label in flip_classes) and (random.random() < config.concept_flip_rate)

        if flipped:
            stream_labels.append(flip_target)
        else:
            stream_labels.append(label)
        stream_data.append(img)
        concept_mask.append(flipped)

    n = len(stream_data)
    truth = _empty_truth(n, true_type='concept')
    truth['concept_mask'] = np.array(concept_mask[:n], dtype=bool)
    # Under concept drift, in_control_mask tracks whether the *assigned* label
    # is still in the in-control set. Flipped samples are "label-wise OOC".
    truth['in_control_mask'] = np.array(
        [(lbl in config.in_control_classes) for lbl in stream_labels[:n]],
        dtype=bool,
    )
    truth['changepoints'].append((changepoint, 'concept'))
    if end_of_transition < n:
        truth['changepoints'].append((end_of_transition, 'concept'))
    print(f"[Concept] Stream: {n} samples, flip {flip_classes} → {flip_target} "
          f"at rate {config.concept_flip_rate}")
    return stream_data, stream_labels, truth


# ──────────────────────────────────────────────────────────────────────────────
# Registry + top-level dispatcher
# ──────────────────────────────────────────────────────────────────────────────

DRIFT_CREATORS = {
    DriftType.SUDDEN:            create_stream_sudden,
    DriftType.GRADUAL:           create_stream_gradual,
    DriftType.INCREMENTAL:       create_stream_incremental,
    DriftType.RECURRING:         create_stream_recurring,
    DriftType.FEATURE_COVARIATE: create_stream_feature_drift,
    DriftType.CONCEPT:           create_stream_concept,
}


def create_drift_stream(all_data, all_labels, config: DriftConfig):
    """Returns (stream_data, stream_labels, drift_truth)."""
    return DRIFT_CREATORS[config.drift_type](all_data, all_labels, config)


# ──────────────────────────────────────────────────────────────────────────────
# Per-window projection of the ground-truth timeline
# ──────────────────────────────────────────────────────────────────────────────

def project_truth_to_windows(drift_truth, windows):
    """Project the per-sample drift_truth onto window indices.

    Returns a dict with:
      'window_indices':         list of window indices
      'ooc_fraction':           fraction of OOC (label-wise) samples per window
      'feature_intensity':      mean augmentation intensity per window
      'concept_flip_fraction':  fraction of label-flipped samples per window
      'drift_intensity':        combined 0-1 drift strength per window
      'first_drift_window':     first window where ANY drift > 0 (None if none)
      'substantial_drift_window': first window where drift_intensity > 0.2
      'changepoint_windows':    list of (window_idx, kind) of true changepoints
    """
    in_ctrl = drift_truth['in_control_mask']
    feat    = drift_truth['feature_intensity']
    concept = drift_truth['concept_mask']

    win_idx, ooc_frac, feat_mean, concept_frac = [], [], [], []
    for w in windows:
        s, e = w['start_idx'], w['end_idx']
        s = max(0, min(s, len(in_ctrl)))
        e = max(0, min(e, len(in_ctrl)))
        if e <= s:
            # empty — pad with zeros
            win_idx.append(w['index'])
            ooc_frac.append(0.0); feat_mean.append(0.0); concept_frac.append(0.0)
            continue
        win_idx.append(w['index'])
        ooc_frac.append(float(np.mean(~in_ctrl[s:e])))
        feat_mean.append(float(np.mean(feat[s:e])))
        concept_frac.append(float(np.mean(concept[s:e])))

    ooc_frac     = np.array(ooc_frac)
    feat_mean    = np.array(feat_mean)
    concept_frac = np.array(concept_frac)
    # Unified drift intensity: max of the three signals
    drift_intensity = np.maximum.reduce([ooc_frac, feat_mean, concept_frac])

    # Find the first window after the baseline where drift appears
    # (windows[0] is the baseline; we skip it for latency metrics)
    non_baseline_mask = np.array([not w['is_baseline'] for w in windows], dtype=bool)
    first_any = None
    first_sub = None
    for i, w in enumerate(windows):
        if w['is_baseline']:
            continue
        if drift_intensity[i] > 0.01 and first_any is None:
            first_any = w['index']
        if drift_intensity[i] > 0.20 and first_sub is None:
            first_sub = w['index']
        if first_any is not None and first_sub is not None:
            break

    # Map true sample-level changepoints to window indices
    cp_windows = []
    for (sample_idx, kind) in drift_truth['changepoints']:
        # find the first window containing sample_idx
        for w in windows:
            if w['is_baseline']:
                continue
            if w['start_idx'] <= sample_idx < w['end_idx']:
                cp_windows.append((w['index'], kind))
                break

    return {
        'window_indices':          win_idx,
        'ooc_fraction':            ooc_frac,
        'feature_intensity':       feat_mean,
        'concept_flip_fraction':   concept_frac,
        'drift_intensity':         drift_intensity,
        'first_drift_window':      first_any,
        'substantial_drift_window': first_sub,
        'changepoint_windows':     cp_windows,
        'true_drift_type':         drift_truth['true_drift_type'],
    }


In [7]:
def create_windows(stream_data, stream_labels,
                   window_size=100, step_size=100, baseline_size=1000):
    windows = [{
        'index': 0, 'start_idx': 0, 'end_idx': baseline_size,
        'data': stream_data[:baseline_size],
        'labels': stream_labels[:baseline_size],
        'is_baseline': True,
    }]
    window_idx = 1
    start_idx  = baseline_size
    n_samples  = len(stream_data)
    while start_idx + window_size <= n_samples:
        end_idx = start_idx + window_size
        windows.append({
            'index': window_idx, 'start_idx': start_idx, 'end_idx': end_idx,
            'data': stream_data[start_idx:end_idx],
            'labels': stream_labels[start_idx:end_idx],
            'is_baseline': False,
        })
        window_idx += 1
        start_idx  += step_size
    print(f"Created {len(windows)} windows "
          f"(baseline={baseline_size}, window={window_size}, step={step_size})")
    return windows


In [8]:
class ResNetFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        weights = ResNet18_Weights.IMAGENET1K_V1
        resnet  = resnet18(weights=weights)
        for param in resnet.parameters():
            param.requires_grad = False
        self.first_conv_block = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1  = resnet.layer1
        self.layer2  = resnet.layer2
        self.layer3  = resnet.layer3
        self.layer4  = resnet.layer4
        self.avgpool = resnet.avgpool
        # Shared Global Average Pooling used to reduce all intermediate
        # spatial feature maps to a compact channel-wise descriptor vector.
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.eval()

    def forward(self, x):
        def gap_vec(t):
            """Apply GAP and flatten: (B, C, H, W) → (B, C)."""
            return torch.flatten(self.gap(t), 1)

        x0  = self.first_conv_block(x)   # (B, 64,  56, 56)
        x1  = self.layer1(x0)            # (B, 64,  56, 56)
        x2  = self.layer2(x1)            # (B, 128, 28, 28)
        x3  = self.layer3(x2)            # (B, 256, 14, 14)
        x4  = self.layer4(x3)            # (B, 512,  7,  7)
        pen = torch.flatten(self.avgpool(x4), 1)  # (B, 512) — same as gap_vec(x4)

        return {
            'layer0':      gap_vec(x0),  # 64-dim  — first conv + maxpool
            'layer1':      gap_vec(x1),  # 64-dim  — residual stage 1
            'layer2':      gap_vec(x2),  # 128-dim — residual stage 2
            'layer3':      gap_vec(x3),  # 256-dim — residual stage 3
            'penultimate': pen,           # 512-dim — residual stage 4 + avgpool
        }


def extract_features_from_window(window_data, feature_extractor, batch_size=64):
    """Extract features for all extractor output levels from a window of images."""
    feature_extractor.eval()
    accum = {}   # level_key → list[np.ndarray]
    with torch.no_grad():
        for i in range(0, len(window_data), batch_size):
            batch = torch.stack(window_data[i:i + batch_size]).to(device)
            feats = feature_extractor(batch)
            for key, tensor in feats.items():
                accum.setdefault(key, []).append(tensor.cpu().numpy())
    return {key: np.vstack(arrs) for key, arrs in accum.items()}


In [9]:
# ── VAE Architecture ──────────────────────────────────────────────────────────
# Variational Autoencoder trained on raw pixel data (denormalized from ImageNet
# normalization to [0,1]).  Used for drift detection via reconstruction error
# and latent KL divergence — operates independently of the ResNet feature
# extractor.

class VAEEncoder(nn.Module):
    """Convolutional encoder: 224x224x3 -> latent (mu, logvar)."""
    def __init__(self, latent_dim: int = 128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1), nn.ReLU(),    # 224 -> 112
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(),   # 112 -> 56
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(),  # 56  -> 28
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1), nn.ReLU(), # 28  -> 14
            nn.Conv2d(256, 256, kernel_size=4, stride=2, padding=1), nn.ReLU(), # 14  -> 7
        )
        self.fc_mu     = nn.Linear(256 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(256 * 7 * 7, latent_dim)

    def forward(self, x):
        h = self.conv(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)


class VAEDecoder(nn.Module):
    """Convolutional decoder: latent -> 224x224x3."""
    def __init__(self, latent_dim: int = 128):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 256 * 7 * 7)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1), nn.ReLU(),  # 7  -> 14
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(),  # 14 -> 28
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(),   # 28 -> 56
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.ReLU(),    # 56 -> 112
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1), nn.Sigmoid(),  # 112 -> 224
        )

    def forward(self, z):
        h = self.fc(z).view(-1, 256, 7, 7)
        return self.deconv(h)


class ConvVAE(nn.Module):
    def __init__(self, latent_dim: int = 128):
        super().__init__()
        self.encoder = VAEEncoder(latent_dim)
        self.decoder = VAEDecoder(latent_dim)

    def reparameterise(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z          = self.reparameterise(mu, logvar)
        recon      = self.decoder(z)
        return recon, mu, logvar

    def vae_loss(self, recon, x, mu, logvar, beta: float = 1.0):
        recon_loss = nn.functional.mse_loss(recon, x, reduction='sum') / x.size(0)
        kld        = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        return recon_loss + beta * kld, recon_loss, kld


class VAEDriftDetector:
    """
    Drift detector based on VAE reconstruction error and latent KL divergence.

    Trained on baseline raw images (denormalized to [0,1] pixel space).
    Two drift signals:
      1. Reconstruction MSE > baseline_mean + n_sigma * baseline_std
      2. Latent KL divergence per window (continuous signal)
    """
    def __init__(self, latent_dim=128, n_epochs=20, batch_size=64,
                 lr=1e-3, beta=1.0, n_sigma=3.0):
        self.latent_dim  = latent_dim
        self.n_epochs    = n_epochs
        self.batch_size  = batch_size
        self.lr          = lr
        self.beta        = beta
        self.n_sigma     = n_sigma

        self.vae:                       Optional[ConvVAE]    = None
        self.baseline_errors:           Optional[np.ndarray] = None
        self.baseline_latent_kl:        Optional[np.ndarray] = None
        self.threshold:                 float = float('inf')
        self.baseline_mean:             float = 0.0
        self.baseline_std:              float = 1.0
        self.baseline_latent_kl_mean:   float = 0.0
        self._fitted: bool = False

    def _to_tensor_batch(self, image_list, batch_size=64):
        """Yield batches of image tensors moved to device."""
        for i in range(0, len(image_list), batch_size):
            yield torch.stack(image_list[i:i + batch_size]).to(device)

    def fit(self, baseline_images: list):
        """Train VAE on baseline raw images (pre-ResNet tensors)."""
        print(f"  [VAE] Training on {len(baseline_images)} baseline images "
              f"({self.n_epochs} epochs, latent_dim={self.latent_dim})...")

        self.vae = ConvVAE(latent_dim=self.latent_dim).to(device)
        optimiser = torch.optim.Adam(self.vae.parameters(), lr=self.lr)

        self.vae.train()
        for epoch in range(self.n_epochs):
            epoch_loss = 0.0
            n_batches  = 0
            indices    = list(range(len(baseline_images)))
            random.shuffle(indices)
            shuffled   = [baseline_images[i] for i in indices]

            for batch in self._to_tensor_batch(shuffled, self.batch_size):
                x = denormalize_imagenet(batch)
                optimiser.zero_grad()
                recon, mu, logvar = self.vae(x)
                loss, _, _        = self.vae.vae_loss(recon, x, mu, logvar, self.beta)
                loss.backward()
                optimiser.step()
                epoch_loss += loss.item()
                n_batches  += 1

            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"    Epoch {epoch+1:3d}/{self.n_epochs} — loss: {epoch_loss/n_batches:.4f}")

        self.baseline_errors, self.baseline_latent_kl = self._compute_errors_and_kl(baseline_images)
        self.baseline_mean           = float(np.mean(self.baseline_errors))
        self.baseline_std            = float(np.std(self.baseline_errors)) + 1e-8
        self.threshold               = self.baseline_mean + self.n_sigma * self.baseline_std
        self.baseline_latent_kl_mean = float(np.mean(self.baseline_latent_kl))
        self._fitted = True
        print(f"  [VAE] Baseline error: mean={self.baseline_mean:.6f}, "
              f"std={self.baseline_std:.6f}, threshold={self.threshold:.6f}")
        print(f"  [VAE] Baseline latent KL: mean={self.baseline_latent_kl_mean:.4f}")

    def _compute_errors_and_kl(self, image_list: list):
        """Return (per-sample MSE, per-sample latent KL) as two numpy arrays."""
        self.vae.eval()
        errors, kls = [], []
        with torch.no_grad():
            for batch in self._to_tensor_batch(image_list, self.batch_size):
                x = denormalize_imagenet(batch)
                recon, mu, logvar = self.vae(x)
                mse = ((recon - x) ** 2).mean(dim=[1, 2, 3])
                latent_kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
                errors.append(mse.cpu().numpy())
                kls.append(latent_kl.cpu().numpy())
        return np.concatenate(errors), np.concatenate(kls)

    def score_window(self, window_images: list) -> dict:
        """Score a window returning reconstruction error, latent KL, and drift flag."""
        errors, latent_kls = self._compute_errors_and_kl(window_images)
        mean_error     = float(np.mean(errors))
        mean_latent_kl = float(np.mean(latent_kls))
        ks_stat, ks_pvalue = ks_2samp(self.baseline_errors, errors)
        return {
            'mean_error':     mean_error,
            'mean_latent_kl': mean_latent_kl,
            'drift_flag':     mean_error > self.threshold,
            'ks_pvalue':      float(ks_pvalue),
            'error_shift':    (mean_error - self.baseline_mean) / self.baseline_std,
            'per_sample_errors': errors,
        }

    def compute_per_sample_errors(self, image_list: list) -> np.ndarray:
        """Return per-sample reconstruction MSE for DDM/EDDM input."""
        errors, _ = self._compute_errors_and_kl(image_list)
        return errors

print("VAE drift detector defined.")

VAE drift detector defined.


In [10]:
# ── ADWIN (Adaptive Windowing) ────────────────────────────────────────────────
# Standalone implementation of the ADWIN algorithm (Bifet & Gavalda, 2007).
# ADWIN maintains a variable-length window of scalar observations and detects
# distributional changes by comparing sub-window means using Hoeffding bounds.

class ADWIN:
    """
    ADWIN (ADaptive WINdowing) change detector.

    Maintains a sliding window of observations and detects distribution changes
    by finding a partition of the window into two sub-windows whose means differ
    by more than a Hoeffding-bound threshold.

    Parameters
    ----------
    delta : float
        Confidence parameter (lower = more sensitive). Default 0.002.
    """
    def __init__(self, delta: float = 0.002):
        self.delta = delta
        self._window: list = []
        self._sum: float = 0.0
        self.drift_detected: bool = False

    def update(self, value: float) -> bool:
        """Add a new observation and check for drift.

        Returns True if drift was detected at this step.
        """
        self._window.append(value)
        self._sum += value
        self.drift_detected = False

        if len(self._window) < 5:
            return False

        n = len(self._window)
        sum_left = 0.0

        for i in range(1, n):
            sum_left += self._window[i - 1]
            n_left  = i
            n_right = n - i

            if n_right < 2:
                break

            mean_left  = sum_left / n_left
            mean_right = (self._sum - sum_left) / n_right
            diff       = abs(mean_left - mean_right)

            m = 1.0 / (1.0 / n_left + 1.0 / n_right)
            epsilon = np.sqrt(np.log(4.0 * n / self.delta) / (2.0 * m))

            if diff >= epsilon:
                self._window = self._window[i:]
                self._sum = sum(self._window)
                self.drift_detected = True
                return True

        return False

    def reset(self):
        self._window = []
        self._sum = 0.0
        self.drift_detected = False


def compute_window_pixel_mean(window_data: list) -> float:
    """Compute mean pixel intensity (denormalized to [0,1]) for a window of images."""
    batch = torch.stack(window_data)
    raw   = denormalize_imagenet(batch)
    return float(raw.mean().item())


def run_adwin_on_sequence(values: list, delta: float = 0.002) -> list:
    """Run ADWIN on a sequence of scalar values (one per window).

    Returns list of bool, one per value — True where ADWIN detected a change.
    """
    adwin = ADWIN(delta=delta)
    drift_flags = []
    for val in values:
        adwin.update(val)
        drift_flags.append(adwin.drift_detected)
    return drift_flags

print("ADWIN drift detector defined.")

ADWIN drift detector defined.


In [11]:
# ── DDM (Drift Detection Method) ──────────────────────────────────────────────
# Gama et al. (2004).  Tracks running error rate and its standard deviation.
# Signals warning when p + s >= p_min + s_min + warning_level * s_min, and
# drift when p + s >= p_min + s_min + drift_level * s_min.
#
# ── EDDM (Early Drift Detection Method) ──────────────────────────────────────
# Baena-Garcia et al. (2006).  Tracks distances between consecutive errors.
# More sensitive to gradual drift than DDM because it detects when errors
# start clustering closer together.
#
# Both use binarized VAE reconstruction error as the proxy error signal:
#   anomalous = 1 if per-sample VAE recon error > threshold, else 0.

class DDM:
    """Drift Detection Method (Gama et al., 2004).

    Parameters
    ----------
    warning_level : float   Number of std deviations for warning zone (default 2.0)
    drift_level   : float   Number of std deviations for drift zone (default 3.0)
    min_samples   : int     Minimum observations before monitoring starts (default 30)
    """
    def __init__(self, warning_level: float = 2.0, drift_level: float = 3.0,
                 min_samples: int = 30):
        self.warning_level = warning_level
        self.drift_level   = drift_level
        self.min_samples   = min_samples
        self.reset()

    def reset(self):
        self.n: int         = 0
        self.p: float       = 0.0      # running error rate
        self.s: float       = 0.0      # running std = sqrt(p*(1-p)/n)
        self.p_min: float   = float('inf')
        self.s_min: float   = float('inf')
        self.in_warning: bool = False

    def update(self, value: int) -> str:
        """Feed a binary observation (0 = normal, 1 = anomalous).

        Returns 'normal', 'warning', or 'drift'.
        """
        self.n += 1
        self.p += (value - self.p) / self.n
        self.s = np.sqrt(self.p * (1 - self.p) / self.n) if self.n > 1 else 0.0

        if self.n < self.min_samples:
            return "normal"

        if self.p + self.s < self.p_min + self.s_min:
            self.p_min = self.p
            self.s_min = self.s

        if self.s_min < 1e-10:
            return "normal"

        if self.p + self.s >= self.p_min + self.s_min + self.drift_level * self.s_min:
            self.reset()
            return "drift"
        elif self.p + self.s >= self.p_min + self.s_min + self.warning_level * self.s_min:
            self.in_warning = True
            return "warning"
        else:
            self.in_warning = False
            return "normal"


class EDDM:
    """Early Drift Detection Method (Baena-Garcia et al., 2006).

    Tracks distances between consecutive errors.  Signals drift when the
    running (mean + 2*std) of inter-error distances drops below a fraction
    of its historical maximum.

    Parameters
    ----------
    alpha       : float   Warning threshold ratio (default 0.95)
    beta        : float   Drift threshold ratio (default 0.90)
    min_samples : int     Minimum errors before monitoring starts (default 30)
    """
    def __init__(self, alpha: float = 0.95, beta: float = 0.90,
                 min_samples: int = 30):
        self.alpha       = alpha
        self.beta        = beta
        self.min_samples = min_samples
        self.reset()

    def reset(self):
        self.n_errors: int      = 0
        self.n_total: int       = 0
        self.last_error_idx: int = 0
        self.mean_dist: float   = 0.0
        self.var_dist: float    = 0.0     # running variance (Welford)
        self.max_metric: float  = 0.0     # max(mean + 2*std) seen
        self.in_warning: bool   = False

    def update(self, value: int) -> str:
        """Feed a binary observation (0 = normal, 1 = anomalous).

        Returns 'normal', 'warning', or 'drift'.
        """
        self.n_total += 1

        if value == 0:
            return "warning" if self.in_warning else "normal"

        # This is an error observation
        self.n_errors += 1
        if self.n_errors < 2:
            self.last_error_idx = self.n_total
            return "normal"

        distance = self.n_total - self.last_error_idx
        self.last_error_idx = self.n_total

        # Welford's online algorithm for mean and variance
        old_mean = self.mean_dist
        self.mean_dist += (distance - self.mean_dist) / (self.n_errors - 1)
        self.var_dist  += (distance - old_mean) * (distance - self.mean_dist)

        if self.n_errors < self.min_samples:
            return "normal"

        std_dist = np.sqrt(self.var_dist / (self.n_errors - 1)) if self.n_errors > 2 else 0.0
        metric = self.mean_dist + 2.0 * std_dist

        if metric > self.max_metric:
            self.max_metric = metric

        if self.max_metric < 1e-10:
            return "normal"

        ratio = metric / self.max_metric

        if ratio < self.beta:
            self.reset()
            return "drift"
        elif ratio < self.alpha:
            self.in_warning = True
            return "warning"
        else:
            self.in_warning = False
            return "normal"


def run_ddm_eddm_on_windows(windows, vae_detector: VAEDriftDetector) -> dict:
    """Run DDM and EDDM on binarized per-sample VAE reconstruction errors.

    Iterates through all windows, computes per-sample VAE errors, binarizes
    against the VAE threshold, and feeds each binary value to DDM and EDDM.

    Returns
    -------
    dict with keys:
      - window_indices: list of int
      - ddm_drift_flags: list of bool (True if DDM flagged drift in that window)
      - ddm_warning_flags: list of bool
      - eddm_drift_flags: list of bool
      - eddm_warning_flags: list of bool
      - per_window_anomaly_rate: list of float
    """
    ddm  = DDM(warning_level=2.0, drift_level=3.0, min_samples=30)
    eddm = EDDM(alpha=0.95, beta=0.90, min_samples=30)

    results = {
        'window_indices':         [],
        'ddm_drift_flags':        [],
        'ddm_warning_flags':      [],
        'eddm_drift_flags':       [],
        'eddm_warning_flags':     [],
        'per_window_anomaly_rate': [],
    }

    for window in tqdm(windows, desc="DDM + EDDM"):
        idx = window['index']
        results['window_indices'].append(idx)

        # Compute per-sample VAE reconstruction errors
        errors = vae_detector.compute_per_sample_errors(window['data'])
        binary = (errors > vae_detector.threshold).astype(int)
        anomaly_rate = float(binary.mean())
        results['per_window_anomaly_rate'].append(anomaly_rate)

        # Feed each sample to DDM and EDDM
        ddm_drifted = False
        ddm_warned  = False
        eddm_drifted = False
        eddm_warned  = False

        for b in binary:
            ddm_state  = ddm.update(int(b))
            eddm_state = eddm.update(int(b))
            if ddm_state == "drift":
                ddm_drifted = True
            elif ddm_state == "warning":
                ddm_warned = True
            if eddm_state == "drift":
                eddm_drifted = True
            elif eddm_state == "warning":
                eddm_warned = True

        # Reset after baseline so monitoring starts fresh
        if window['is_baseline']:
            ddm.reset()
            eddm.reset()

        results['ddm_drift_flags'].append(ddm_drifted)
        results['ddm_warning_flags'].append(ddm_warned)
        results['eddm_drift_flags'].append(eddm_drifted)
        results['eddm_warning_flags'].append(eddm_warned)

    n_total = len(results['ddm_drift_flags']) - 1
    n_ddm   = sum(results['ddm_drift_flags'][1:])
    n_eddm  = sum(results['eddm_drift_flags'][1:])
    print(f"  [DDM]  Drift detected: {n_ddm}/{n_total} windows")
    print(f"  [EDDM] Drift detected: {n_eddm}/{n_total} windows")

    return results

print("DDM and EDDM drift detectors defined.")

DDM and EDDM drift detectors defined.


In [12]:
def process_all_windows_vae_adwin(windows, vae_detector: VAEDriftDetector,
                                  adwin_delta: float = 0.002) -> dict:
    """
    Run VAE and ADWIN drift detection over all windows.

    If the VAE detector is already fitted, it is reused (no retraining).
    Otherwise, it is fitted on the baseline window.

    ADWIN is run post-hoc on the collected per-window scalars (mean pixel
    intensity and VAE reconstruction error).

    Returns a dict with per-window metrics and binary ADWIN drift flags.
    """
    vae_results = {
        'window_indices': [],
        'vae_recon_error': [],
        'vae_latent_kl': [],
        'vae_error_shift': [],
        'vae_drift_flag': [],
        'vae_ks_pvalue': [],
        'vae_threshold': None,
        'vae_baseline_mean': None,
        'pixel_means': [],
    }

    for window in tqdm(windows, desc="VAE + pixel stats"):
        idx = window['index']

        pixel_mean = compute_window_pixel_mean(window['data'])
        vae_results['pixel_means'].append(pixel_mean)

        if window['is_baseline']:
            if not vae_detector._fitted:
                vae_detector.fit(window['data'])
            vae_results['vae_threshold']     = vae_detector.threshold
            vae_results['vae_baseline_mean'] = vae_detector.baseline_mean
            vae_results['window_indices'].append(idx)
            vae_results['vae_recon_error'].append(vae_detector.baseline_mean)
            vae_results['vae_latent_kl'].append(vae_detector.baseline_latent_kl_mean)
            vae_results['vae_error_shift'].append(0.0)
            vae_results['vae_drift_flag'].append(False)
            vae_results['vae_ks_pvalue'].append(1.0)
        else:
            score = vae_detector.score_window(window['data'])
            vae_results['window_indices'].append(idx)
            vae_results['vae_recon_error'].append(score['mean_error'])
            vae_results['vae_latent_kl'].append(score['mean_latent_kl'])
            vae_results['vae_error_shift'].append(score['error_shift'])
            vae_results['vae_drift_flag'].append(score['drift_flag'])
            vae_results['vae_ks_pvalue'].append(score['ks_pvalue'])

    # Run ADWIN on the collected per-window sequences
    vae_results['adwin_pixel_drift'] = run_adwin_on_sequence(
        vae_results['pixel_means'], delta=adwin_delta)
    vae_results['adwin_vae_drift'] = run_adwin_on_sequence(
        vae_results['vae_recon_error'], delta=adwin_delta)

    n_total     = len(vae_results['vae_drift_flag']) - 1
    n_vae       = sum(vae_results['vae_drift_flag'][1:])
    n_adwin_pix = sum(vae_results['adwin_pixel_drift'][1:])
    n_adwin_vae = sum(vae_results['adwin_vae_drift'][1:])
    print(f"  [VAE]         Drift flagged:  {n_vae}/{n_total} windows")
    print(f"  [ADWIN-pixel] Drift detected: {n_adwin_pix}/{n_total} windows")
    print(f"  [ADWIN-VAE]   Drift detected: {n_adwin_vae}/{n_total} windows")

    return vae_results

In [13]:
print("Initialising ResNet-18 feature extractor...")
feature_extractor = ResNetFeatureExtractor().to(device)
print("Feature extractor ready.")


Initialising ResNet-18 feature extractor...
Feature extractor ready.


In [14]:
def fit_pca(features, n_components=None, variance_retention=0.80, max_components=None):
    """Fit PCA with optional hard cap on number of components.

    Parameters
    ----------
    max_components : int or None
        Hard upper bound.  Typical rule of thumb: window_size // 5 so that
        the sample-to-dimension ratio n/p >= 5 for stable covariance estimation.
    """
    if n_components is None:
        pca = PCA(n_components=variance_retention, svd_solver='full')
        pca.fit(features)
        if max_components is not None and pca.n_components_ > max_components:
            pca = PCA(n_components=max_components, svd_solver='full')
            pca.fit(features)
    else:
        effective = min(n_components, max_components) if max_components else n_components
        pca = PCA(n_components=effective)
        pca.fit(features)
    return pca


def transform_with_pca(features, pca):
    return pca.transform(features)


In [15]:
def compute_kl_divergence_gaussian(mean_p, cov_p, mean_q, cov_q):
    """KL(P ‖ Q) for multivariate Gaussians."""
    k   = len(mean_p)
    # Scale-relative regularisation: 1% of average variance per dimension.
    # This is more robust than a fixed eps=1e-6 when window covariance estimates
    # are noisy (small window size relative to number of PCA components).
    eps_p = max(1e-6, np.trace(cov_p) / k * 0.01)
    eps_q = max(1e-6, np.trace(cov_q) / k * 0.01)
    cov_p_reg = cov_p + eps_p * np.eye(k)
    cov_q_reg = cov_q + eps_q * np.eye(k)
    try:
        cov_q_inv          = np.linalg.inv(cov_q_reg)
        _, logdet_p        = np.linalg.slogdet(cov_p_reg)
        s_q, logdet_q      = np.linalg.slogdet(cov_q_reg)
        if s_q <= 0:
            return np.nan
        trace_term = np.trace(cov_q_inv @ cov_p_reg)
        diff       = mean_q - mean_p
        mahal      = diff @ cov_q_inv @ diff
        return 0.5 * (trace_term + mahal - k + logdet_q - logdet_p)
    except np.linalg.LinAlgError:
        return np.nan


def compute_jsd_gaussian(mean_p, cov_p, mean_q, cov_q):
    """Jensen-Shannon Divergence (symmetric) — approximated as mean of forward + reverse KL."""
    kl_pq = compute_kl_divergence_gaussian(mean_p, cov_p, mean_q, cov_q)
    kl_qp = compute_kl_divergence_gaussian(mean_q, cov_q, mean_p, cov_p)
    if np.isnan(kl_pq) or np.isnan(kl_qp):
        return np.nan
    return 0.5 * (kl_pq + kl_qp)


def compute_hotelling_t2(baseline_scores, window_scores):
    """Hotelling's T²: tests whether two multivariate samples share the same mean."""
    n1, n2   = len(baseline_scores), len(window_scores)
    mean1    = np.mean(baseline_scores, axis=0)
    mean2    = np.mean(window_scores,   axis=0)
    cov1     = np.cov(baseline_scores,  rowvar=False)
    cov2     = np.cov(window_scores,    rowvar=False)
    pooled   = ((n1 - 1) * cov1 + (n2 - 1) * cov2) / (n1 + n2 - 2)
    pooled  += 1e-6 * np.eye(pooled.shape[0])
    try:
        pooled_inv = np.linalg.inv(pooled)
        diff       = mean1 - mean2
        return (n1 * n2) / (n1 + n2) * (diff @ pooled_inv @ diff)
    except np.linalg.LinAlgError:
        return np.nan


def compute_psi(baseline_scores, window_scores, n_bins=10):
    """Population Stability Index: per-component, then summed."""
    total_psi   = 0.0
    n_components = baseline_scores.shape[1]
    for j in range(n_components):
        edges = np.percentile(baseline_scores[:, j], np.linspace(0, 100, n_bins + 1))
        edges[0]  = -np.inf
        edges[-1] =  np.inf
        base_counts = np.histogram(baseline_scores[:, j], bins=edges)[0].astype(float)
        win_counts  = np.histogram(window_scores[:, j],   bins=edges)[0].astype(float)
        base_pct = (base_counts + 1) / (base_counts.sum() + n_bins)
        win_pct  = (win_counts  + 1) / (win_counts.sum()  + n_bins)
        total_psi += np.sum((win_pct - base_pct) * np.log(win_pct / base_pct))
    return total_psi


def compute_energy_distance(baseline_scores, window_scores, max_samples=500):
    """Energy distance: non-parametric two-sample statistic (subsampled for speed)."""
    n1  = min(len(baseline_scores), max_samples)
    n2  = min(len(window_scores),   max_samples)
    idx1 = np.random.choice(len(baseline_scores), n1, replace=False)
    idx2 = np.random.choice(len(window_scores),   n2, replace=False)
    X, Y = baseline_scores[idx1], window_scores[idx2]
    d_xy = cdist(X, Y, 'euclidean').mean()
    d_xx = cdist(X, X, 'euclidean').mean()
    d_yy = cdist(Y, Y, 'euclidean').mean()
    return 2 * d_xy - d_xx - d_yy


In [ ]:

# =============================================================================
# NEW CELL C: Enhanced 4-way drift classifier + detection latency metrics
# =============================================================================
# Drop-in replacement for the existing `classify_drift_type` cell (Cell 16),
# plus latency helpers for the comparison stage.

from scipy.stats import ks_2samp
import numpy as np


# ──────────────────────────────────────────────────────────────────────────────
# Four-way drift classifier
# ──────────────────────────────────────────────────────────────────────────────
# Classifies the *observed* drift (from statistical signals) into one of:
#
#   'none'       no drift detected in either labels or features
#   'label'      P(Y) shifted; feature distribution stable     (prior shift)
#   'feature'    P(X) shifted; labels stable                   (covariate)
#   'concept'    P(X) and P(Y) both shifted, i.e. the label-feature
#                relationship changed. In this synthetic setup concept drift
#                is injected by label-flipping, so we signal it when the
#                feature distribution looks stable relative to baseline BUT
#                the label distribution has changed AND there's a conditional
#                signal — in practice we detect "concept" when label changed
#                AND features changed together in a correlated way.
#   'data'       general / ambiguous drift (fallback)
#
# In practice in this CIFAR setup, "label + feature" drift (e.g. switching
# class mix) is the closest thing to real-world concept drift, so we treat
# concurrent label+feature signal as concept drift.
# ──────────────────────────────────────────────────────────────────────────────

def classify_drift_type(baseline_labels, window_labels,
                         baseline_scores, window_scores, in_control_classes,
                         prior_threshold: float = 0.15,
                         ks_alpha: float = 0.01,
                         ks_fraction: float = 0.5):
    """Four-way classification: none / label / feature / concept (+ data).

    Parameters
    ----------
    prior_threshold : float
        Absolute change in IC fraction to call a label/prior shift.
    ks_alpha : float
        Per-dimension p-value threshold for the KS test.
    ks_fraction : float
        Fraction of tested PCA dimensions that must reject H0 for a
        covariate/feature shift to be signalled.
    """
    # ── 1. Prior / label-side signal ──────────────────────────────────────
    base_in_pct = sum(1 for l in baseline_labels if l in in_control_classes) / max(len(baseline_labels), 1)
    win_in_pct  = sum(1 for l in window_labels   if l in in_control_classes) / max(len(window_labels),   1)
    prior_delta = abs(base_in_pct - win_in_pct)
    prior_shift = prior_delta > prior_threshold

    # Additional label signal: per-class histogram distance (Jensen-Shannon)
    # Captures incremental / recurring mixes that keep IC fraction stable
    # but rebalance OOC classes.
    all_classes = sorted(set(baseline_labels) | set(window_labels))
    if len(all_classes) > 0:
        b_hist = np.array([sum(1 for l in baseline_labels if l == c) for c in all_classes], dtype=float)
        w_hist = np.array([sum(1 for l in window_labels   if l == c) for c in all_classes], dtype=float)
        b_hist = b_hist / max(b_hist.sum(), 1)
        w_hist = w_hist / max(w_hist.sum(), 1)
        m = 0.5 * (b_hist + w_hist)
        def _kl_safe(p, q, eps=1e-10):
            p = p + eps; q = q + eps
            return float(np.sum(p * np.log(p / q)))
        label_jsd = 0.5 * _kl_safe(b_hist, m) + 0.5 * _kl_safe(w_hist, m)
    else:
        label_jsd = 0.0
    # Secondary label signal: treat modest JSD (>=0.02) as a weak prior shift
    prior_shift = prior_shift or (label_jsd > 0.02)

    # ── 2. Feature / covariate signal (KS tests on PCA dimensions) ────────
    n_test = min(5, baseline_scores.shape[1]) if baseline_scores.ndim == 2 else 0
    if n_test > 0:
        ks_pvalues = [ks_2samp(baseline_scores[:, j], window_scores[:, j])[1]
                      for j in range(n_test)]
        feature_reject_frac = np.mean([p < ks_alpha for p in ks_pvalues])
        covariate_shift = feature_reject_frac > ks_fraction
    else:
        ks_pvalues = []
        feature_reject_frac = 0.0
        covariate_shift = False

    # ── 3. Combine into a four-way label ──────────────────────────────────
    #  * label only          → "Label drift (prior shift)"
    #  * feature only        → "Feature drift (covariate shift)"
    #  * both                → "Concept drift (P(Y|X) change)"
    #  * neither             → "No drift detected"
    if prior_shift and covariate_shift:
        kind = "concept"
        label = "Concept drift (P(Y|X) change)"
    elif prior_shift and not covariate_shift:
        kind = "label"
        label = "Label drift (prior shift)"
    elif covariate_shift and not prior_shift:
        kind = "feature"
        label = "Feature drift (covariate shift)"
    else:
        kind = "none"
        label = "No significant drift detected"

    return {
        'kind':                kind,        # machine-readable category
        'label':               label,       # human-readable
        'prior_shift':         bool(prior_shift),
        'covariate_shift':     bool(covariate_shift),
        'base_in_pct':         float(base_in_pct),
        'win_in_pct':          float(win_in_pct),
        'prior_delta':         float(prior_delta),
        'label_jsd':           float(label_jsd),
        'ks_pvalues':          ks_pvalues,
        'feature_reject_frac': float(feature_reject_frac),
    }


# ──────────────────────────────────────────────────────────────────────────────
# Detection latency metrics
# ──────────────────────────────────────────────────────────────────────────────

def _first_true_window_index(flags, window_indices, skip_baseline=True):
    """Return the window index of the first True flag (skipping baseline, i=0)."""
    start = 1 if skip_baseline else 0
    for i in range(start, len(flags)):
        if flags[i]:
            return window_indices[i]
    return None


def compute_detection_latency(results, thresholds, vae_results, ddm_eddm_results,
                              truth_windows):
    """Compute detection latency for every framework relative to the
    ground-truth first-drift and substantial-drift windows.

    Positive latency  = detection is LATE (after true drift)
    Negative latency  = detection is EARLY (false-alarm before drift)
    None              = method never fired

    Returns a dict: method_name -> {
        'detection_window': int or None,
        'latency_vs_first_drift': int or None,
        'latency_vs_substantial_drift': int or None,
    }
    """
    first = truth_windows['first_drift_window']
    sub   = truth_windows['substantial_drift_window']

    def _mk(detect_win):
        lat_first = (detect_win - first) if (detect_win is not None and first is not None) else None
        lat_sub   = (detect_win - sub)   if (detect_win is not None and sub   is not None) else None
        return {
            'detection_window':             detect_win,
            'latency_vs_first_drift':       lat_first,
            'latency_vs_substantial_drift': lat_sub,
        }

    metrics = {}

    # ── ResNet-based (per feature level) ──────────────────────────────────
    for level in results:
        win_idx = results[level]['window_indices']
        for key, pretty in [('kl_significant',  f'ResNet-KL ({level})'),
                             ('jsd_significant', f'ResNet-JSD ({level})')]:
            dw = _first_true_window_index(results[level][key], win_idx)
            metrics[pretty] = _mk(dw)

    # ── VAE + ADWIN ───────────────────────────────────────────────────────
    if vae_results is not None:
        vw = vae_results['window_indices']
        for key, pretty in [('vae_drift_flag',      'VAE-recon-error'),
                             ('adwin_pixel_drift',  'ADWIN (pixel mean)'),
                             ('adwin_vae_drift',    'ADWIN (VAE error)')]:
            if key in vae_results:
                dw = _first_true_window_index(vae_results[key], vw)
                metrics[pretty] = _mk(dw)

    # ── DDM / EDDM ────────────────────────────────────────────────────────
    if ddm_eddm_results is not None:
        dw_idx = ddm_eddm_results['window_indices']
        for key, pretty in [('ddm_drift_flags',  'DDM'),
                             ('eddm_drift_flags', 'EDDM')]:
            if key in ddm_eddm_results:
                dw = _first_true_window_index(ddm_eddm_results[key], dw_idx)
                metrics[pretty] = _mk(dw)

    return metrics


def summarize_window_classifications(results, level='penultimate'):
    """Count how the per-window 4-way classifier labelled each post-baseline
    window at the given level."""
    kinds = {'none': 0, 'label': 0, 'feature': 0, 'concept': 0, 'data': 0}
    cls_list = results[level]['drift_classification']
    for i, c in enumerate(cls_list):
        if i == 0:
            continue  # baseline
        # c can be either the new dict-backed label string or a legacy string
        lower = c.lower() if isinstance(c, str) else ''
        if 'no significant' in lower or c == 'Baseline':
            kinds['none'] += 1
        elif 'concept' in lower:
            kinds['concept'] += 1
        elif 'label drift' in lower or 'prior' in lower:
            kinds['label'] += 1
        elif 'feature' in lower or 'covariate' in lower:
            kinds['feature'] += 1
        else:
            kinds['data'] += 1
    return kinds


In [17]:
def process_all_windows(windows, feature_extractor, in_control_classes,
                         variance_retention=0.80, alpha=0.05, n_bootstrap=200):
    """Process all windows and compute multiple drift metrics.

    Thresholds for KL and JSD are calibrated empirically by bootstrapping
    sub-windows of the same size as the test windows from the baseline scores.
    This accounts for finite-sample covariance estimation noise and avoids the
    chi-squared approximation (which assumes large n and ignores window-size
    effects on the Gaussian KL estimator).

    Returns
    -------
    results    : dict of per-level metric lists
    thresholds : dict  {level: {'kl': float, 'jsd': float}}
    """
    levels  = ['layer0', 'layer1', 'layer2', 'layer3', 'penultimate']
    results = {
        level: {
            'window_indices': [], 'kl_vs_baseline': [], 'jsd_vs_baseline': [],
            'kl_consecutive': [], 'hotelling_t2': [], 'psi': [],
            'energy_distance': [], 'kl_significant': [], 'jsd_significant': [],
            'drift_classification': [], 'window_means': [], 'window_covs': [],
        }
        for level in levels
    }

    baseline_models = {}
    thresholds      = {}
    prev_scores     = {}
    prev_mean       = {}
    prev_cov        = {}

    # Derive test window size from the first non-baseline window.
    window_size = (windows[1]['end_idx'] - windows[1]['start_idx']
                   if len(windows) > 1 else 100)

    for window in tqdm(windows, desc="Processing windows"):
        window_idx = window['index']
        features   = extract_features_from_window(window['data'], feature_extractor, batch_size=64)

        for level in levels:
            level_features = features[level]

            if window['is_baseline']:
                pca    = fit_pca(level_features, variance_retention=variance_retention)
                scores = transform_with_pca(level_features, pca)
                mean   = np.mean(scores, axis=0)
                cov    = np.cov(scores, rowvar=False)

                # ------------------------------------------------------------------
                # Bootstrap threshold calibration
                # Draw n_bootstrap sub-windows of size window_size from the baseline
                # scores (already in PCA space) to empirically measure how large KL
                # and JSD can be purely due to sampling variability.
                # ------------------------------------------------------------------
                n_baseline = len(scores)
                rng = np.random.default_rng(42)
                boot_kl, boot_jsd = [], []
                for _ in range(n_bootstrap):
                    idx     = rng.choice(n_baseline, size=window_size, replace=False)
                    b_sc    = scores[idx]
                    b_mean  = np.mean(b_sc, axis=0)
                    b_cov   = np.cov(b_sc, rowvar=False)
                    bkl  = compute_kl_divergence_gaussian(mean, cov, b_mean, b_cov)
                    bjsd = compute_jsd_gaussian(mean, cov, b_mean, b_cov)
                    if not np.isnan(bkl):
                        boot_kl.append(bkl)
                    if not np.isnan(bjsd):
                        boot_jsd.append(bjsd)

                kl_threshold  = np.percentile(boot_kl,  (1 - alpha) * 100)
                jsd_threshold = np.percentile(boot_jsd, (1 - alpha) * 100)

                baseline_models[level] = {'pca': pca, 'mean': mean, 'cov': cov, 'scores': scores}
                thresholds[level]      = {'kl': kl_threshold, 'jsd': jsd_threshold}
                prev_scores[level]     = scores
                prev_mean[level]       = mean
                prev_cov[level]        = cov

                results[level]['window_indices'].append(window_idx)
                for key in ('kl_vs_baseline', 'jsd_vs_baseline', 'kl_consecutive',
                            'hotelling_t2', 'psi', 'energy_distance'):
                    results[level][key].append(0.0)
                results[level]['kl_significant'].append(False)
                results[level]['jsd_significant'].append(False)
                results[level]['drift_classification'].append("Baseline")
                results[level]['window_means'].append(mean)
                results[level]['window_covs'].append(cov)

            else:
                bm     = baseline_models[level]
                scores = transform_with_pca(level_features, bm['pca'])
                mean   = np.mean(scores, axis=0)
                cov    = np.cov(scores, rowvar=False)

                kl_base  = compute_kl_divergence_gaussian(bm['mean'], bm['cov'], mean, cov)
                jsd_base = compute_jsd_gaussian(bm['mean'], bm['cov'], mean, cov)
                kl_consec = compute_kl_divergence_gaussian(
                    prev_mean[level], prev_cov[level], mean, cov)
                t2     = compute_hotelling_t2(bm['scores'], scores)
                psi    = compute_psi(bm['scores'], scores)
                energy = compute_energy_distance(bm['scores'], scores)

                sig_kl  = (not np.isnan(kl_base))  and kl_base  > thresholds[level]['kl']
                sig_jsd = (not np.isnan(jsd_base)) and jsd_base > thresholds[level]['jsd']
                drift_cls = classify_drift_type(
                    windows[0]['labels'], window['labels'],
                    bm['scores'], scores, in_control_classes)

                results[level]['window_indices'].append(window_idx)
                results[level]['kl_vs_baseline'].append(kl_base)
                results[level]['jsd_vs_baseline'].append(jsd_base)
                results[level]['kl_consecutive'].append(kl_consec)
                results[level]['hotelling_t2'].append(t2)
                results[level]['psi'].append(psi)
                results[level]['energy_distance'].append(energy)
                results[level]['kl_significant'].append(sig_kl)
                results[level]['jsd_significant'].append(sig_jsd)
                results[level]['drift_classification'].append(drift_cls['label'])
                results[level]['window_means'].append(mean)
                results[level]['window_covs'].append(cov)

                prev_scores[level] = scores
                prev_mean[level]   = mean
                prev_cov[level]    = cov

    return results, thresholds


In [18]:
LAYER_LABELS = {
    'layer0':      'Layer 0 — first conv + maxpool (64-dim GAP)',
    'layer1':      'Layer 1 — residual stage 1 (64-dim GAP)',
    'layer2':      'Layer 2 — residual stage 2 (128-dim GAP)',
    'layer3':      'Layer 3 — residual stage 3 (256-dim GAP)',
    'penultimate': 'Penultimate — residual stage 4 + avgpool (512-dim)',
}


def plot_drift_dashboard(results, thresholds, drift_type_name: str,
                         output_dir='./output', level='penultimate'):
    """6-panel dashboard of all drift metrics for a single feature level."""
    os.makedirs(output_dir, exist_ok=True)
    fig, axes = plt.subplots(3, 2, figsize=(18, 16))
    layer_label = LAYER_LABELS.get(level, level)
    fig.suptitle(
        f'Concept Drift Detection Dashboard — {drift_type_name}\n'
        f'Feature level: {layer_label}',
        fontsize=14, fontweight='bold', y=0.98)

    win_idx    = results[level]['window_indices']
    kl_thresh  = thresholds[level]['kl']
    jsd_thresh = thresholds[level]['jsd']
    metrics = [
        ('kl_vs_baseline',   'KL Divergence vs Baseline',             'steelblue',   kl_thresh),
        ('jsd_vs_baseline',  'Jensen-Shannon Divergence vs Baseline',  'darkorange',  jsd_thresh),
        ('kl_consecutive',   'KL Divergence (Consecutive Windows)',     'purple',      None),
        ('hotelling_t2',     "Hotelling's T² vs Baseline",             'darkred',     None),
        ('psi',              'Population Stability Index (PSI)',        'teal',        None),
        ('energy_distance',  'Energy Distance vs Baseline',            'forestgreen', None),
    ]

    for ax, (key, title, color, threshold) in zip(axes.flatten(), metrics):
        values = results[level][key]
        ax.plot(win_idx, values, marker='.', markersize=3, linewidth=1.5, color=color)
        if threshold is not None:
            ax.axhline(y=threshold, color='red', linestyle='--', linewidth=1.5,
                       alpha=0.7, label='Threshold (α=0.05, bootstrap)')
            ax.legend(fontsize=8)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xlabel('Window Index')
        ax.grid(True, alpha=0.3, linestyle=':')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    safe_name = drift_type_name.lower().replace(" ", "_").replace("/", "_")
    fname = os.path.join(output_dir, f'drift_dashboard_{safe_name}_{level}.png')
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    print(f"Dashboard saved to {fname}")
    plt.show()
    plt.close()


def plot_layer_kl_comparison(results, thresholds, drift_type_name: str,
                              output_dir='./output'):
    """One subplot per feature layer showing KL divergence vs baseline and its threshold."""
    os.makedirs(output_dir, exist_ok=True)
    levels = list(results.keys())
    n      = len(levels)
    colors_kl  = ['steelblue', 'royalblue', 'mediumblue', 'navy', 'darkslateblue']
    colors_jsd = ['darkorange', 'orangered', 'firebrick', 'maroon', 'saddlebrown']

    fig, axes = plt.subplots(n, 1, figsize=(14, 3.5 * n), sharex=False)
    if n == 1:
        axes = [axes]
    fig.suptitle(
        f'KL & JSD Divergence by Feature Layer — {drift_type_name}',
        fontsize=14, fontweight='bold', y=1.01)

    for ax, level, ckl, cjsd in zip(axes, levels, colors_kl, colors_jsd):
        win_idx  = results[level]['window_indices']
        kl_vals  = results[level]['kl_vs_baseline']
        jsd_vals = results[level]['jsd_vs_baseline']
        kl_thr   = thresholds[level]['kl']
        jsd_thr  = thresholds[level]['jsd']

        ax.plot(win_idx, kl_vals,  marker='.', markersize=2, linewidth=1.4,
                color=ckl,  label='KL vs baseline')
        ax.plot(win_idx, jsd_vals, marker='.', markersize=2, linewidth=1.4,
                color=cjsd, label='JSD vs baseline', linestyle='--', alpha=0.8)
        ax.axhline(y=kl_thr,  color=ckl,  linestyle=':',  linewidth=1.5,
                   alpha=0.9, label=f'KL threshold  ({kl_thr:.3f})')
        ax.axhline(y=jsd_thr, color=cjsd, linestyle=':', linewidth=1.5,
                   alpha=0.9, label=f'JSD threshold ({jsd_thr:.3f})')

        layer_label = LAYER_LABELS.get(level, level)
        ax.set_title(layer_label, fontsize=10, fontweight='bold')
        ax.set_ylabel('Divergence')
        ax.legend(fontsize=7, ncol=2)
        ax.grid(True, alpha=0.3, linestyle=':')

    axes[-1].set_xlabel('Window Index')
    plt.tight_layout()
    safe_name = drift_type_name.lower().replace(" ", "_").replace("/", "_")
    fname = os.path.join(output_dir, f'layer_kl_comparison_{safe_name}.png')
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    print(f"Layer KL comparison saved to {fname}")
    plt.show()
    plt.close()


def plot_method_comparison_dashboard(results, thresholds, vae_results, ddm_eddm_results,
                                     drift_type_name: str, output_dir='./output',
                                     level='penultimate'):
    """3x3 dashboard comparing all drift detection methods side-by-side."""
    os.makedirs(output_dir, exist_ok=True)
    fig, axes = plt.subplots(3, 3, figsize=(24, 18))
    fig.suptitle(
        f'All Methods Comparison — {drift_type_name}',
        fontsize=15, fontweight='bold', y=0.98)

    win_idx_resnet = results[level]['window_indices']
    win_idx_vae    = vae_results['window_indices']
    win_idx_ddm    = ddm_eddm_results['window_indices']

    # Row 1: ResNet-based metrics
    ax = axes[0, 0]
    ax.plot(win_idx_resnet, results[level]['kl_vs_baseline'],
            marker='.', markersize=2, linewidth=1.4, color='steelblue')
    ax.axhline(y=thresholds[level]['kl'], color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.set_title('KL Divergence (ResNet)', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle=':')

    ax = axes[0, 1]
    ax.plot(win_idx_resnet, results[level]['jsd_vs_baseline'],
            marker='.', markersize=2, linewidth=1.4, color='darkorange')
    ax.axhline(y=thresholds[level]['jsd'], color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.set_title('JSD (ResNet)', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle=':')

    ax = axes[0, 2]
    ax.plot(win_idx_resnet, results[level]['hotelling_t2'],
            marker='.', markersize=2, linewidth=1.4, color='darkred')
    ax.set_title("Hotelling's T² (ResNet)", fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle=':')

    # Row 2: VAE and ADWIN
    ax = axes[1, 0]
    ax.plot(win_idx_vae, vae_results['vae_recon_error'],
            marker='.', markersize=2, linewidth=1.4, color='mediumorchid')
    if vae_results['vae_threshold'] is not None:
        ax.axhline(y=vae_results['vae_threshold'], color='red', linestyle='--',
                   linewidth=1.5, alpha=0.7, label='VAE threshold')
        ax.legend(fontsize=7)
    ax.set_title('VAE Reconstruction Error (raw pixels)', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle=':')

    ax = axes[1, 1]
    ax.plot(win_idx_vae, vae_results['vae_latent_kl'],
            marker='.', markersize=2, linewidth=1.4, color='darkviolet')
    ax.set_title('VAE Latent KL (raw pixels)', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle=':')

    ax = axes[1, 2]
    # ADWIN as step plot (binary drift flags)
    adwin_pix = [int(d) for d in vae_results['adwin_pixel_drift']]
    adwin_vae = [int(d) for d in vae_results['adwin_vae_drift']]
    ax.step(win_idx_vae, adwin_pix, where='post', linewidth=1.5, color='teal',
            label='ADWIN (pixel mean)', alpha=0.8)
    ax.step(win_idx_vae, adwin_vae, where='post', linewidth=1.5, color='coral',
            label='ADWIN (VAE error)', alpha=0.8)
    ax.set_ylim(-0.1, 1.5)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['No Drift', 'Drift'])
    ax.set_title('ADWIN Drift Detection (raw data)', fontsize=10, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3, linestyle=':')

    # Row 3: DDM, EDDM, anomaly rate
    ax = axes[2, 0]
    ddm_flags = [int(d) for d in ddm_eddm_results['ddm_drift_flags']]
    ddm_warn  = [int(d) for d in ddm_eddm_results['ddm_warning_flags']]
    ax.step(win_idx_ddm, ddm_flags, where='post', linewidth=1.5, color='crimson',
            label='DDM drift', alpha=0.9)
    ax.step(win_idx_ddm, ddm_warn, where='post', linewidth=1.2, color='orange',
            label='DDM warning', alpha=0.6, linestyle='--')
    ax.set_ylim(-0.1, 1.5)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['No Drift', 'Drift'])
    ax.set_title('DDM (VAE error proxy)', fontsize=10, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3, linestyle=':')

    ax = axes[2, 1]
    eddm_flags = [int(d) for d in ddm_eddm_results['eddm_drift_flags']]
    eddm_warn  = [int(d) for d in ddm_eddm_results['eddm_warning_flags']]
    ax.step(win_idx_ddm, eddm_flags, where='post', linewidth=1.5, color='darkgreen',
            label='EDDM drift', alpha=0.9)
    ax.step(win_idx_ddm, eddm_warn, where='post', linewidth=1.2, color='limegreen',
            label='EDDM warning', alpha=0.6, linestyle='--')
    ax.set_ylim(-0.1, 1.5)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['No Drift', 'Drift'])
    ax.set_title('EDDM (VAE error proxy)', fontsize=10, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3, linestyle=':')

    ax = axes[2, 2]
    ax.plot(win_idx_ddm, ddm_eddm_results['per_window_anomaly_rate'],
            marker='.', markersize=2, linewidth=1.4, color='saddlebrown')
    ax.set_title('Per-Window Anomaly Rate (VAE)', fontsize=10, fontweight='bold')
    ax.set_ylabel('Fraction anomalous')
    ax.grid(True, alpha=0.3, linestyle=':')

    for ax in axes.flatten():
        ax.set_xlabel('Window Index')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    safe_name = drift_type_name.lower().replace(" ", "_").replace("/", "_")
    fname = os.path.join(output_dir, f'all_methods_{safe_name}.png')
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    print(f"All-methods dashboard saved to {fname}")
    plt.show()
    plt.close()


def plot_drift_type_comparison(all_results, all_thresholds, output_dir='./output',
                                level='penultimate'):
    """Compare KL divergence across different drift injection types for one feature level."""
    os.makedirs(output_dir, exist_ok=True)
    n = len(all_results)
    fig, axes = plt.subplots(n, 1, figsize=(14, 4 * n), sharex=False)
    if n == 1:
        axes = [axes]

    colors = plt.cm.tab10(np.linspace(0, 1, max(n, 10)))

    for ax, ((name, results), color) in zip(axes, zip(all_results.items(), colors)):
        win_idx = results[level]['window_indices']
        kl_vals = results[level]['kl_vs_baseline']
        ax.plot(win_idx, kl_vals, marker='.', markersize=3, linewidth=1.5, color=color)
        if name in all_thresholds and level in all_thresholds[name]:
            kl_thresh = all_thresholds[name][level]['kl']
            ax.axhline(y=kl_thresh, color='red', linestyle='--',
                       linewidth=1.5, alpha=0.7, label='KL threshold (α=0.05, bootstrap)')
            ax.legend(fontsize=8)
        ax.set_title(name, fontsize=12, fontweight='bold')
        ax.set_ylabel('KL Divergence')
        ax.grid(True, alpha=0.3, linestyle=':')

    axes[-1].set_xlabel('Window Index')
    plt.tight_layout()
    fname = os.path.join(output_dir, f'drift_type_comparison_{level}.png')
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    print(f"Comparison plot saved to {fname}")
    plt.show()
    plt.close()


def plot_vae_comparison(all_vae_results, output_dir='./output'):
    """Compare VAE reconstruction error across drift types."""
    os.makedirs(output_dir, exist_ok=True)
    n = len(all_vae_results)
    if n == 0:
        return
    fig, axes = plt.subplots(n, 1, figsize=(14, 4 * n), sharex=False)
    if n == 1:
        axes = [axes]

    colors = plt.cm.tab10(np.linspace(0, 1, max(n, 10)))

    for ax, ((name, vr), color) in zip(axes, zip(all_vae_results.items(), colors)):
        ax.plot(vr['window_indices'], vr['vae_recon_error'],
                marker='.', markersize=3, linewidth=1.5, color=color)
        if vr['vae_threshold'] is not None:
            ax.axhline(y=vr['vae_threshold'], color='red', linestyle='--',
                       linewidth=1.5, alpha=0.7, label='VAE threshold')
            ax.legend(fontsize=8)
        ax.set_title(f'{name} — VAE Reconstruction Error', fontsize=12, fontweight='bold')
        ax.set_ylabel('MSE')
        ax.grid(True, alpha=0.3, linestyle=':')

    axes[-1].set_xlabel('Window Index')
    plt.tight_layout()
    fname = os.path.join(output_dir, 'vae_drift_type_comparison.png')
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    print(f"VAE comparison saved to {fname}")
    plt.show()
    plt.close()


def print_drift_summary(results, thresholds, drift_name, in_control_classes,
                         vae_results=None, ddm_eddm_results=None):
    """Print a tabular summary of drift detection results for every feature level."""
    print(f"\n{'='*70}")
    print(f"DRIFT SUMMARY: {drift_name}")
    print(f"{'='*70}")
    for level in results:
        kl_sig  = results[level]['kl_significant']
        jsd_sig = results[level]['jsd_significant']
        n_kl    = sum(kl_sig[1:])
        n_jsd   = sum(jsd_sig[1:])
        n_total = len(kl_sig) - 1
        layer_label = LAYER_LABELS.get(level, level)
        print(f"\n  [{layer_label}]")
        print(f"    KL threshold (bootstrap, α=0.05):  {thresholds[level]['kl']:.4f}")
        print(f"    JSD threshold (bootstrap, α=0.05): {thresholds[level]['jsd']:.4f}")
        print(f"    KL-significant drift windows:  {n_kl}/{n_total}")
        print(f"    JSD-significant drift windows: {n_jsd}/{n_total}")
        print(f"    Max KL vs baseline:        {np.nanmax(results[level]['kl_vs_baseline']):.4f}")
        print(f"    Max JSD vs baseline:       {np.nanmax(results[level]['jsd_vs_baseline']):.4f}")
        print(f"    Max Hotelling T²:          {np.nanmax(results[level]['hotelling_t2']):.4f}")
        print(f"    Max PSI:                   {np.nanmax(results[level]['psi']):.4f}")
        print(f"    Max Energy Distance:       {np.nanmax(results[level]['energy_distance']):.4f}")
        for i, s in enumerate(kl_sig):
            if s:
                print(f"    First KL drift at:         window {results[level]['window_indices'][i]}")
                break
        for i, s in enumerate(jsd_sig):
            if s:
                print(f"    First JSD drift at:        window {results[level]['window_indices'][i]}")
                break
        last_cls = results[level]['drift_classification'][-1]
        print(f"    Drift type (last window):  {last_cls}")

    # VAE / ADWIN / DDM / EDDM summary
    if vae_results is not None:
        n_total_v = len(vae_results['vae_drift_flag']) - 1
        print(f"\n  [VAE — raw pixel data]")
        print(f"    Threshold:         {vae_results['vae_threshold']:.6f}")
        print(f"    Drift windows:     {sum(vae_results['vae_drift_flag'][1:])}/{n_total_v}")
        print(f"    Max recon error:   {max(vae_results['vae_recon_error']):.6f}")
        print(f"    Max latent KL:     {max(vae_results['vae_latent_kl']):.4f}")
        print(f"    ADWIN-pixel drift: {sum(vae_results['adwin_pixel_drift'][1:])}/{n_total_v}")
        print(f"    ADWIN-VAE drift:   {sum(vae_results['adwin_vae_drift'][1:])}/{n_total_v}")

    if ddm_eddm_results is not None:
        n_total_d = len(ddm_eddm_results['ddm_drift_flags']) - 1
        print(f"\n  [DDM / EDDM — VAE error proxy]")
        print(f"    DDM drift windows:   {sum(ddm_eddm_results['ddm_drift_flags'][1:])}/{n_total_d}")
        print(f"    DDM warning windows: {sum(ddm_eddm_results['ddm_warning_flags'][1:])}/{n_total_d}")
        print(f"    EDDM drift windows:  {sum(ddm_eddm_results['eddm_drift_flags'][1:])}/{n_total_d}")
        print(f"    EDDM warning windows:{sum(ddm_eddm_results['eddm_warning_flags'][1:])}/{n_total_d}")
        # First detection indices
        for i, d in enumerate(ddm_eddm_results['ddm_drift_flags']):
            if d:
                print(f"    First DDM drift at:  window {ddm_eddm_results['window_indices'][i]}")
                break
        for i, d in enumerate(ddm_eddm_results['eddm_drift_flags']):
            if d:
                print(f"    First EDDM drift at: window {ddm_eddm_results['window_indices'][i]}")
                break

In [ ]:

# =============================================================================
# NEW CELL D: Ground-truth overlay plots + latency summary plots
# =============================================================================
# Insert AFTER the existing plotting cell (Cell 18). Provides new versions of
# the per-experiment and cross-experiment dashboards that overlay the
# ground-truth drift timeline on every detection plot, so you can visually
# assess how early/late each method fires.

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


def _overlay_truth(ax, truth_windows, colors=('lightgray', 'gold', 'tomato'),
                   show_cp=True, alpha_fill=0.22, label_prefix=''):
    """Fill the background with the ground-truth drift intensity on a twin axis
    and mark changepoints with vertical lines on the main axis.

    - Out-of-control fraction is plotted as a grey area (0..1)
    - Feature-augmentation intensity is a gold area (0..1)
    - Concept-flip fraction is a tomato area (0..1)
    All overlays share the same right-axis (0..1) so they can be compared
    to each other. The main detection metric stays on the left y-axis.
    """
    wi    = truth_windows['window_indices']
    ooc   = truth_windows['ooc_fraction']
    feat  = truth_windows['feature_intensity']
    cpt   = truth_windows['concept_flip_fraction']

    ax_r = ax.twinx()
    if ooc.max() > 0.01:
        ax_r.fill_between(wi, 0, ooc, color=colors[0],  alpha=alpha_fill, step='mid',
                          label=f'{label_prefix}OOC fraction')
    if feat.max() > 0.01:
        ax_r.fill_between(wi, 0, feat, color=colors[1], alpha=alpha_fill, step='mid',
                          label=f'{label_prefix}Feature intensity')
    if cpt.max() > 0.01:
        ax_r.fill_between(wi, 0, cpt, color=colors[2],  alpha=alpha_fill, step='mid',
                          label=f'{label_prefix}Concept flip frac')
    ax_r.set_ylim(0, 1.05)
    ax_r.set_ylabel('Ground-truth drift (0–1)', fontsize=8, color='dimgray')
    ax_r.tick_params(axis='y', labelsize=7, colors='dimgray')

    if show_cp:
        for (cp_w, kind) in truth_windows['changepoint_windows']:
            clr = {'label': 'gray', 'feature': 'darkgoldenrod',
                   'concept': 'firebrick'}.get(kind, 'black')
            ax.axvline(cp_w, color=clr, linestyle='-', linewidth=1.0,
                       alpha=0.55, zorder=0)

    return ax_r


def _mark_first_detection(ax, detection_window, color='red', label='first detection'):
    if detection_window is None:
        return
    ax.axvline(detection_window, color=color, linestyle=':', linewidth=1.8,
               alpha=0.9, zorder=1)


def plot_drift_dashboard_with_truth(results, thresholds, truth_windows,
                                     drift_type_name: str, output_dir='./output',
                                     level='penultimate'):
    """6-panel dashboard of ResNet drift metrics, each overlaid on the
    ground-truth drift timeline. Vertical dotted red line marks the first
    significant detection for KL / JSD."""
    os.makedirs(output_dir, exist_ok=True)
    fig, axes = plt.subplots(3, 2, figsize=(18, 16))
    layer_label = LAYER_LABELS.get(level, level)
    fig.suptitle(
        f'Drift Detection Dashboard — {drift_type_name}\n'
        f'Feature level: {layer_label}   '
        f'(background shading = ground-truth drift timeline)',
        fontsize=14, fontweight='bold', y=0.99)

    win_idx    = results[level]['window_indices']
    kl_thresh  = thresholds[level]['kl']
    jsd_thresh = thresholds[level]['jsd']

    kl_first  = next((win_idx[i] for i, s in enumerate(results[level]['kl_significant']) if i > 0 and s), None)
    jsd_first = next((win_idx[i] for i, s in enumerate(results[level]['jsd_significant']) if i > 0 and s), None)

    metrics = [
        ('kl_vs_baseline',   'KL Divergence vs Baseline',             'steelblue',   kl_thresh,  kl_first),
        ('jsd_vs_baseline',  'Jensen-Shannon Divergence vs Baseline',  'darkorange',  jsd_thresh, jsd_first),
        ('kl_consecutive',   'KL Divergence (Consecutive Windows)',     'purple',      None,      None),
        ('hotelling_t2',     "Hotelling's T² vs Baseline",             'darkred',     None,      None),
        ('psi',              'Population Stability Index (PSI)',        'teal',        None,      None),
        ('energy_distance',  'Energy Distance vs Baseline',            'forestgreen', None,      None),
    ]

    for ax, (key, title, color, threshold, first_det) in zip(axes.flatten(), metrics):
        _overlay_truth(ax, truth_windows)
        values = results[level][key]
        ax.plot(win_idx, values, marker='.', markersize=3, linewidth=1.6,
                color=color, zorder=3)
        if threshold is not None:
            ax.axhline(y=threshold, color='red', linestyle='--', linewidth=1.3,
                       alpha=0.7, zorder=2)
        _mark_first_detection(ax, first_det)

        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xlabel('Window Index')
        ax.grid(True, alpha=0.25, linestyle=':')

        # Compose legend
        handles = [Line2D([], [], color=color, marker='.', linewidth=1.6,
                          label=title.split('(')[0].strip())]
        if threshold is not None:
            handles.append(Line2D([], [], color='red', linestyle='--',
                                   label=f'Threshold ({threshold:.3f})'))
        if first_det is not None:
            handles.append(Line2D([], [], color='red', linestyle=':',
                                   label=f'First detection (win {first_det})'))
        ax.legend(handles=handles, fontsize=7, loc='upper left')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    safe_name = drift_type_name.lower().replace(" ", "_").replace("/", "_")
    fname = os.path.join(output_dir, f'drift_dashboard_truth_{safe_name}_{level}.png')
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    print(f"Dashboard (with truth overlay) saved to {fname}")
    plt.show()
    plt.close()


def plot_method_comparison_with_truth(results, thresholds, vae_results,
                                       ddm_eddm_results, truth_windows,
                                       drift_type_name: str, output_dir='./output',
                                       level='penultimate'):
    """3x3 comparison of all methods with ground-truth overlay on each subplot.
    First-detection for each method is drawn as a dotted red vertical line."""
    os.makedirs(output_dir, exist_ok=True)
    fig, axes = plt.subplots(3, 3, figsize=(24, 18))
    fig.suptitle(
        f'All Methods vs Ground Truth — {drift_type_name}\n'
        f'(background shading = true drift timeline, dotted red = first detection)',
        fontsize=15, fontweight='bold', y=0.99)

    win_r = results[level]['window_indices']
    win_v = vae_results['window_indices']
    win_d = ddm_eddm_results['window_indices']

    def _first_true(flags, idxs):
        for i in range(1, len(flags)):
            if flags[i]:
                return idxs[i]
        return None

    kl_first  = _first_true(results[level]['kl_significant'],  win_r)
    jsd_first = _first_true(results[level]['jsd_significant'], win_r)
    vae_first = _first_true(vae_results['vae_drift_flag'], win_v) if 'vae_drift_flag' in vae_results else None
    apx_first = _first_true(vae_results['adwin_pixel_drift'], win_v)
    avae_first = _first_true(vae_results['adwin_vae_drift'],   win_v)
    ddm_first = _first_true(ddm_eddm_results['ddm_drift_flags'], win_d)
    eddm_first = _first_true(ddm_eddm_results['eddm_drift_flags'], win_d)

    panels = [
        # (row, col, x, y, title, color, threshold, first_det, kind)
        (0, 0, win_r, results[level]['kl_vs_baseline'],  'KL Divergence (ResNet)',
         'steelblue',  thresholds[level]['kl'],  kl_first,  'line'),
        (0, 1, win_r, results[level]['jsd_vs_baseline'], 'JSD (ResNet)',
         'darkorange', thresholds[level]['jsd'], jsd_first, 'line'),
        (0, 2, win_r, results[level]['hotelling_t2'],    "Hotelling's T² (ResNet)",
         'darkred',   None,                      None,     'line'),
        (1, 0, win_v, vae_results['vae_recon_error'],    'VAE Reconstruction Error',
         'mediumorchid', vae_results.get('vae_threshold'), vae_first, 'line'),
        (1, 1, win_v, vae_results['vae_latent_kl'],      'VAE Latent KL',
         'darkviolet',    None, None, 'line'),
        (1, 2, win_v, [int(d) for d in vae_results['adwin_pixel_drift']],
         'ADWIN (raw pixel mean)',   'teal',      None,  apx_first, 'step'),
        (2, 0, win_v, [int(d) for d in vae_results['adwin_vae_drift']],
         'ADWIN (VAE error)',        'coral',     None,  avae_first, 'step'),
        (2, 1, win_d, [int(d) for d in ddm_eddm_results['ddm_drift_flags']],
         'DDM (VAE-error proxy)',    'crimson',   None,  ddm_first, 'step'),
        (2, 2, win_d, [int(d) for d in ddm_eddm_results['eddm_drift_flags']],
         'EDDM (VAE-error proxy)',   'darkgreen', None,  eddm_first, 'step'),
    ]

    for (r, c, xs, ys, title, color, thresh, first_det, kind) in panels:
        ax = axes[r, c]
        _overlay_truth(ax, truth_windows)
        if kind == 'line':
            ax.plot(xs, ys, marker='.', markersize=2, linewidth=1.4, color=color, zorder=3)
            if thresh is not None:
                ax.axhline(y=thresh, color='red', linestyle='--', linewidth=1.2,
                           alpha=0.7, zorder=2)
        else:  # step (binary)
            ax.step(xs, ys, where='post', linewidth=1.5, color=color, alpha=0.9, zorder=3)
            ax.set_ylim(-0.1, 1.5)
            ax.set_yticks([0, 1]); ax.set_yticklabels(['No Drift', 'Drift'])
        _mark_first_detection(ax, first_det)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_xlabel('Window Index')
        ax.grid(True, alpha=0.25, linestyle=':')

        # Detection lag annotation
        if first_det is not None and truth_windows['first_drift_window'] is not None:
            lag = first_det - truth_windows['first_drift_window']
            lag_sub = None
            if truth_windows['substantial_drift_window'] is not None:
                lag_sub = first_det - truth_windows['substantial_drift_window']
            annot = f'Δfirst={lag:+d} win'
            if lag_sub is not None:
                annot += f'\nΔsubst={lag_sub:+d} win'
            ax.text(0.98, 0.98, annot, transform=ax.transAxes,
                    fontsize=8, ha='right', va='top',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                              edgecolor='red', alpha=0.85))
        elif first_det is None:
            ax.text(0.98, 0.98, 'no detection', transform=ax.transAxes,
                    fontsize=8, ha='right', va='top',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                              edgecolor='gray', alpha=0.85))

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    safe_name = drift_type_name.lower().replace(" ", "_").replace("/", "_")
    fname = os.path.join(output_dir, f'all_methods_truth_{safe_name}.png')
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    print(f"All-methods-vs-truth dashboard saved to {fname}")
    plt.show()
    plt.close()


def plot_resnet_drift_classification_with_truth(results, truth_windows,
                                                 drift_type_name: str,
                                                 output_dir='./output',
                                                 level='penultimate'):
    """Timeline plot showing, per window, the 4-way classification produced
    by classify_drift_type (none/label/feature/concept), overlaid on the true
    drift intensity."""
    os.makedirs(output_dir, exist_ok=True)
    win_idx = results[level]['window_indices']
    cls_list = results[level]['drift_classification']

    # Encode to integer category for scatter plotting
    cat_map = {'Baseline': 0, 'No significant drift detected': 0,
               'Label drift (prior shift)': 1,
               'Feature drift (covariate shift)': 2,
               'Concept drift (P(Y|X) change)': 3}
    cats = [cat_map.get(c, 0) for c in cls_list]
    palette = {0: 'lightgray', 1: 'steelblue', 2: 'darkorange', 3: 'firebrick'}
    legend_labels = {0: 'none / baseline', 1: 'label (prior)',
                     2: 'feature (covariate)', 3: 'concept P(Y|X)'}

    fig, ax = plt.subplots(figsize=(14, 4.5))
    _overlay_truth(ax, truth_windows)

    for cat_id in [0, 1, 2, 3]:
        xs = [wi for wi, c in zip(win_idx, cats) if c == cat_id]
        ys = [cat_id] * len(xs)
        if xs:
            ax.scatter(xs, ys, c=palette[cat_id], s=32, label=legend_labels[cat_id],
                       edgecolors='black', linewidths=0.3, zorder=4)

    ax.set_yticks([0, 1, 2, 3])
    ax.set_yticklabels(['none', 'label', 'feature', 'concept'])
    ax.set_ylabel('Per-window classification')
    ax.set_xlabel('Window Index')
    ax.set_title(
        f'ResNet-KL 4-way classification per window — {drift_type_name}\n'
        f'(ground truth = "{truth_windows["true_drift_type"]}")',
        fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='upper left', ncol=4)
    ax.grid(True, alpha=0.25, linestyle=':')

    plt.tight_layout()
    safe_name = drift_type_name.lower().replace(" ", "_").replace("/", "_")
    fname = os.path.join(output_dir, f'classification_timeline_{safe_name}_{level}.png')
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    print(f"Classification timeline saved to {fname}")
    plt.show()
    plt.close()


def plot_latency_summary(latency_metrics, truth_windows, drift_type_name: str,
                          output_dir='./output'):
    """Bar chart of first-detection latency (relative to first true drift) for
    every method. Positive = late, negative = early false-alarm."""
    os.makedirs(output_dir, exist_ok=True)

    names  = list(latency_metrics.keys())
    lat_f  = [latency_metrics[n]['latency_vs_first_drift']        for n in names]
    lat_s  = [latency_metrics[n]['latency_vs_substantial_drift']  for n in names]

    # Replace None with NaN
    lat_f_plot = [np.nan if v is None else v for v in lat_f]
    lat_s_plot = [np.nan if v is None else v for v in lat_s]

    fig, axes = plt.subplots(1, 2, figsize=(16, max(4, 0.3 * len(names))))
    for ax, lats, title in zip(axes,
                                [lat_f_plot, lat_s_plot],
                                ['Latency vs FIRST drift onset',
                                 'Latency vs SUBSTANTIAL drift (>20%)']):
        y_pos = np.arange(len(names))
        colors = ['crimson' if (v is not None and not np.isnan(v) and v < 0)
                  else ('steelblue' if (v is not None and not np.isnan(v) and v >= 0)
                        else 'lightgray')
                  for v in lats]
        ax.barh(y_pos, [0 if np.isnan(v) else v for v in lats],
                color=colors, edgecolor='black', linewidth=0.5)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_yticks(y_pos); ax.set_yticklabels(names, fontsize=8)
        ax.set_xlabel('windows (+ = late, − = early / false alarm)')
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.25, linestyle=':', axis='x')
        # Annotate missing detections
        for i, v in enumerate(lats):
            if np.isnan(v):
                ax.text(0.01, i, ' no detection', va='center', fontsize=7,
                        color='dimgray')

    fig.suptitle(f'Detection latency — {drift_type_name}',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    safe_name = drift_type_name.lower().replace(" ", "_").replace("/", "_")
    fname = os.path.join(output_dir, f'latency_{safe_name}.png')
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    print(f"Latency summary saved to {fname}")
    plt.show()
    plt.close()


def plot_drift_type_comparison_with_truth(all_results, all_thresholds, all_truth,
                                           output_dir='./output',
                                           level='penultimate'):
    """Cross-experiment KL plot with each subplot overlaid on its own
    ground-truth timeline."""
    os.makedirs(output_dir, exist_ok=True)
    n = len(all_results)
    fig, axes = plt.subplots(n, 1, figsize=(14, 3.7 * n), sharex=False)
    if n == 1:
        axes = [axes]
    colors = plt.cm.tab10(np.linspace(0, 1, max(n, 10)))

    for ax, ((name, results), color) in zip(axes, zip(all_results.items(), colors)):
        if name in all_truth:
            _overlay_truth(ax, all_truth[name])
        win_idx = results[level]['window_indices']
        kl_vals = results[level]['kl_vs_baseline']
        ax.plot(win_idx, kl_vals, marker='.', markersize=3, linewidth=1.5,
                color=color, zorder=3)
        if name in all_thresholds and level in all_thresholds[name]:
            kl_thresh = all_thresholds[name][level]['kl']
            ax.axhline(y=kl_thresh, color='red', linestyle='--', linewidth=1.3,
                       alpha=0.7, zorder=2)
            # First detection
            first = next((win_idx[i] for i, s in enumerate(results[level]['kl_significant'])
                          if i > 0 and s), None)
            _mark_first_detection(ax, first)
        true_type = all_truth[name]['true_drift_type'] if name in all_truth else 'unknown'
        ax.set_title(f'{name}  (ground-truth: {true_type})',
                     fontsize=12, fontweight='bold')
        ax.set_ylabel('KL Divergence')
        ax.grid(True, alpha=0.3, linestyle=':')

    axes[-1].set_xlabel('Window Index')
    plt.tight_layout()
    fname = os.path.join(output_dir, f'drift_type_comparison_truth_{level}.png')
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    print(f"Comparison plot (with truth) saved to {fname}")
    plt.show()
    plt.close()


def plot_latency_heatmap(all_latency, all_truth, output_dir='./output'):
    """Cross-experiment heatmap of detection latency (relative to first true
    drift). Rows = methods, columns = experiments."""
    os.makedirs(output_dir, exist_ok=True)
    # Collect union of method names
    method_names = []
    for exp, lm in all_latency.items():
        for m in lm:
            if m not in method_names:
                method_names.append(m)
    exp_names = list(all_latency.keys())
    if not exp_names or not method_names:
        print("Not enough data for latency heatmap.")
        return

    # Build latency matrix (rows=methods, cols=experiments)
    mat_first = np.full((len(method_names), len(exp_names)), np.nan)
    mat_sub   = np.full((len(method_names), len(exp_names)), np.nan)
    for j, exp in enumerate(exp_names):
        for i, m in enumerate(method_names):
            entry = all_latency[exp].get(m, None)
            if entry is None:
                continue
            if entry['latency_vs_first_drift'] is not None:
                mat_first[i, j] = entry['latency_vs_first_drift']
            if entry['latency_vs_substantial_drift'] is not None:
                mat_sub[i, j]   = entry['latency_vs_substantial_drift']

    fig, axes = plt.subplots(1, 2, figsize=(max(10, 1.1 * len(exp_names) + 8),
                                             max(6, 0.35 * len(method_names))))
    vmax = max(np.nanmax(np.abs(mat_first)) if np.any(~np.isnan(mat_first)) else 1,
               np.nanmax(np.abs(mat_sub))   if np.any(~np.isnan(mat_sub))   else 1)

    for ax, mat, title in zip(axes,
                               [mat_first, mat_sub],
                               ['Latency vs FIRST drift',
                                'Latency vs SUBSTANTIAL drift']):
        im = ax.imshow(mat, aspect='auto', cmap='RdBu_r',
                       vmin=-vmax, vmax=vmax)
        ax.set_xticks(np.arange(len(exp_names)))
        ax.set_xticklabels(exp_names, rotation=45, ha='right', fontsize=8)
        ax.set_yticks(np.arange(len(method_names)))
        ax.set_yticklabels(method_names, fontsize=8)
        ax.set_title(title, fontsize=11, fontweight='bold')
        # Annotate each cell
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                v = mat[i, j]
                if np.isnan(v):
                    ax.text(j, i, '–', ha='center', va='center', fontsize=7,
                            color='dimgray')
                else:
                    ax.text(j, i, f'{int(v):+d}', ha='center', va='center',
                            fontsize=7,
                            color='white' if abs(v) > vmax * 0.5 else 'black')
        fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02,
                     label='windows (+ late, − early)')

    fig.suptitle('Cross-experiment detection latency (windows vs ground truth)',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    fname = os.path.join(output_dir, 'latency_heatmap_cross_experiment.png')
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    print(f"Latency heatmap saved to {fname}")
    plt.show()
    plt.close()


## Experiment Configuration

Edit `IN_CONTROL` / `OUT_OF_CONTROL` class lists and the `drift_configs` dict below to change what is tested.

CIFAR-10 class map: `0=airplane, 1=automobile, 2=bird, 3=cat, 4=deer, 5=dog, 6=frog, 7=horse, 8=ship, 9=truck`


In [ ]:

# =============================================================================
# NEW CELL E: Drift experiment configurations
# =============================================================================
# Drop-in replacement for Cell 20.
#
# In this CIFAR setup, "concept drift" is NOT injected as a dedicated scenario
# — CIFAR labels are fixed, so true P(Y|X) change isn't naturally expressible.
# Instead, the 4-way ResNet-KL classifier will label windows where BOTH the
# label distribution AND the feature distribution shift as concept drift.
# That means the pure label-drift and pure feature-drift experiments below
# are expected to be classified as 'label' and 'feature' respectively, while
# (if you extend the config later) any experiment that combines a class-mix
# change with augmentations would be classified as concept drift.

IN_CONTROL      = [0]
OUT_OF_CONTROL  = [1, 2, 3, 4, 5, 6, 7, 8, 9]

drift_configs = {
    # ── Label / prior drift (class-mix changes) ────────────────────────────
    "Sudden Drift (label)": DriftConfig(
        drift_type=DriftType.SUDDEN,
        in_control_classes=IN_CONTROL,
        out_of_control_classes=OUT_OF_CONTROL,
        changepoint_frac=0.3,
    ),
    "Gradual Drift (label)": DriftConfig(
        drift_type=DriftType.GRADUAL,
        in_control_classes=IN_CONTROL,
        out_of_control_classes=OUT_OF_CONTROL,
    ),
    "Incremental Drift (label)": DriftConfig(
        drift_type=DriftType.INCREMENTAL,
        in_control_classes=IN_CONTROL,
        out_of_control_classes=OUT_OF_CONTROL,
    ),
    "Recurring Drift (label)": DriftConfig(
        drift_type=DriftType.RECURRING,
        in_control_classes=IN_CONTROL,
        out_of_control_classes=OUT_OF_CONTROL,
        cycle_length=2000,
        n_cycles=4,
    ),

    # ── Feature / covariate drift (augmentations) ──────────────────────────
    "Feature Drift (Noise+Brightness)": DriftConfig(
        drift_type=DriftType.FEATURE_COVARIATE,
        in_control_classes=IN_CONTROL,
        out_of_control_classes=OUT_OF_CONTROL,
        augmentation_types=['noise', 'brightness'],
        noise_std=0.5,
        brightness_shift=0.4,
    ),
    "Feature Drift (Saturation)": DriftConfig(
        drift_type=DriftType.FEATURE_COVARIATE,
        in_control_classes=IN_CONTROL,
        out_of_control_classes=OUT_OF_CONTROL,
        augmentation_types=['saturation'],
        saturation_factor=2.0,
    ),
    "Feature Drift (Contrast)": DriftConfig(
        drift_type=DriftType.FEATURE_COVARIATE,
        in_control_classes=IN_CONTROL,
        out_of_control_classes=OUT_OF_CONTROL,
        augmentation_types=['contrast'],
        contrast_factor=0.3,
    ),
    "Feature Drift (Blur)": DriftConfig(
        drift_type=DriftType.FEATURE_COVARIATE,
        in_control_classes=IN_CONTROL,
        out_of_control_classes=OUT_OF_CONTROL,
        augmentation_types=['blur'],
        blur_kernel_size=7,
        blur_sigma=3.0,
    ),
    "Feature Drift (JPEG Artifacts)": DriftConfig(
        drift_type=DriftType.FEATURE_COVARIATE,
        in_control_classes=IN_CONTROL,
        out_of_control_classes=OUT_OF_CONTROL,
        augmentation_types=['jpeg'],
        jpeg_quality=5,
    ),
    "Feature Drift (Elastic Distortion)": DriftConfig(
        drift_type=DriftType.FEATURE_COVARIATE,
        in_control_classes=IN_CONTROL,
        out_of_control_classes=OUT_OF_CONTROL,
        augmentation_types=['elastic'],
        elastic_alpha=60.0,
        elastic_sigma=5.0,
    ),
    "Feature Drift (All Augmentations)": DriftConfig(
        drift_type=DriftType.FEATURE_COVARIATE,
        in_control_classes=IN_CONTROL,
        out_of_control_classes=OUT_OF_CONTROL,
        augmentation_types=None,
    ),

    # ── Example: concurrent label + feature shift (would classify as concept) ──
    # Uncomment to add a "concept-like" scenario (label change + augmentations).
    # "Concept-like (label + feature)": DriftConfig(
    #     drift_type=DriftType.FEATURE_COVARIATE,
    #     in_control_classes=IN_CONTROL,
    #     out_of_control_classes=OUT_OF_CONTROL,
    #     augmentation_types=['noise', 'brightness'],
    # ),
}


In [ ]:

# =============================================================================
# NEW CELL F: Experiment runner with ground-truth overlays + latency metrics
# =============================================================================
# Drop-in replacement for Cell 21.

all_results        = {}
all_thresholds     = {}
all_vae_results    = {}
all_ddm_eddm       = {}
all_truth_windows  = {}     # NEW — per-experiment projected ground-truth
all_latency        = {}     # NEW — per-experiment per-method latency

# VAE detector — trained once on the first experiment's baseline, reused for all
vae_detector = VAEDriftDetector(latent_dim=64, n_epochs=50, batch_size=64,
                                lr=1e-3, beta=4.0, n_sigma=2.0)

for name, config in drift_configs.items():
    print(f"\n{'#'*70}")
    print(f"# EXPERIMENT: {name}")
    print(f"{'#'*70}")

    set_seeds(42)
    # NOTE: create_drift_stream now returns a third value — the ground-truth
    # drift timeline — describing per-sample in-control / feature-intensity /
    # concept-flip status.
    stream_data, stream_labels, drift_truth = create_drift_stream(
        all_data, all_labels, config)
    windows = create_windows(stream_data, stream_labels,
                              window_size=100, step_size=100, baseline_size=1000)
    truth_windows = project_truth_to_windows(drift_truth, windows)

    # 1. ResNet feature pipeline (KL, JSD, T², PSI, energy distance, 4-way classification)
    results, thresholds = process_all_windows(
        windows, feature_extractor, IN_CONTROL,
        variance_retention=0.95, alpha=0.001, n_bootstrap=500)

    # 2. VAE + ADWIN (raw pixel data)
    vae_adwin_results = process_all_windows_vae_adwin(windows, vae_detector)

    # 3. DDM + EDDM (binarized VAE reconstruction error)
    ddm_eddm_results = run_ddm_eddm_on_windows(windows, vae_detector)

    # 4. Latency of every method vs. ground-truth drift onset
    latency = compute_detection_latency(
        results, thresholds, vae_adwin_results, ddm_eddm_results, truth_windows)

    # Store
    all_results[name]        = results
    all_thresholds[name]     = thresholds
    all_vae_results[name]    = vae_adwin_results
    all_ddm_eddm[name]       = ddm_eddm_results
    all_truth_windows[name]  = truth_windows
    all_latency[name]        = latency

    # 5. Visualisations — all overlaid on the ground-truth timeline
    plot_drift_dashboard_with_truth(results, thresholds, truth_windows,
                                     name, level='penultimate')
    plot_layer_kl_comparison(results, thresholds, name)     # layer view (unchanged)
    plot_method_comparison_with_truth(results, thresholds,
                                       vae_adwin_results, ddm_eddm_results,
                                       truth_windows, name)
    plot_resnet_drift_classification_with_truth(results, truth_windows, name)
    plot_latency_summary(latency, truth_windows, name)

    print_drift_summary(results, thresholds, name, IN_CONTROL,
                         vae_results=vae_adwin_results,
                         ddm_eddm_results=ddm_eddm_results)

    # Text summary of latency (kind-aware)
    print(f"\n  [Ground truth: {truth_windows['true_drift_type']}]")
    print(f"    First drift window (any):         {truth_windows['first_drift_window']}")
    print(f"    First substantial (>20%) window:  {truth_windows['substantial_drift_window']}")
    print(f"    True changepoint windows:         {truth_windows['changepoint_windows']}")

    # ResNet-KL classification accuracy vs true kind
    kinds = summarize_window_classifications(results, level='penultimate')
    total_post = sum(kinds.values())
    if total_post > 0:
        true_kind = truth_windows['true_drift_type']
        correct   = kinds.get(true_kind, 0)
        print(f"    ResNet-KL 4-way classification (penultimate):")
        for k, v in kinds.items():
            marker = ' *' if k == true_kind else ''
            print(f"      {k:>8}: {v:4d} windows{marker}")
        print(f"    Agreement with ground truth ({true_kind}): {correct}/{total_post} "
              f"= {100*correct/total_post:.1f}%")

    del stream_data, stream_labels, windows, drift_truth
    gc.collect()


In [ ]:

# =============================================================================
# NEW CELL G: Cross-experiment comparison with truth + latency heatmap
# =============================================================================
# Drop-in replacement for Cell 22.

# Cross-experiment plots with ground-truth overlays
plot_drift_type_comparison_with_truth(all_results, all_thresholds,
                                       all_truth_windows, level='penultimate')
plot_vae_comparison(all_vae_results)
plot_latency_heatmap(all_latency, all_truth_windows)

# ── Tabular latency summary ──────────────────────────────────────────────────
print("\n" + "=" * 90)
print("DETECTION LATENCY SUMMARY (windows relative to true first drift)")
print("=" * 90)
print(f"{'Experiment':<40} {'True type':<10} {'First drift @ win':<18}")
print("-" * 90)
for name, tw in all_truth_windows.items():
    print(f"  {name:<38} {tw['true_drift_type']:<10} "
          f"first={tw['first_drift_window']}  substantial={tw['substantial_drift_window']}")

print("\n" + "-" * 90)
print("Per-method first-detection latency (windows). +=late, −=early/false-alarm, '—'=no detection")
print("-" * 90)
# Get method set
method_names = []
for exp, lm in all_latency.items():
    for m in lm:
        if m not in method_names:
            method_names.append(m)

col_w = max(12, max(len(n) for n in all_latency.keys()) + 2)
header = f"{'Method':<30} " + "".join(f"{n:<{col_w}}" for n in all_latency.keys())
print(header)
for m in method_names:
    row = f"{m:<30} "
    for exp in all_latency:
        entry = all_latency[exp].get(m, None)
        if entry is None or entry['latency_vs_first_drift'] is None:
            s = '—'
        else:
            s = f"{entry['latency_vs_first_drift']:+d}"
        row += f"{s:<{col_w}}"
    print(row)

# ── Classifier-agreement table (ResNet-KL 4-way vs ground truth) ────────────
print("\n" + "-" * 90)
print("ResNet-KL 4-way classification agreement with ground truth (penultimate layer)")
print("-" * 90)
print(f"{'Experiment':<40} {'True':<10} {'none':>7} {'label':>7} {'feature':>8} {'concept':>8} {'Agreement':>12}")
for name, results in all_results.items():
    kinds  = summarize_window_classifications(results, level='penultimate')
    true_k = all_truth_windows[name]['true_drift_type']
    total  = sum(kinds.values())
    agree  = kinds.get(true_k, 0)
    pct    = 100 * agree / total if total > 0 else 0.0
    print(f"  {name:<38} {true_k:<10} {kinds['none']:>7} {kinds['label']:>7} "
          f"{kinds['feature']:>8} {kinds['concept']:>8} {agree}/{total} ({pct:.1f}%)")

print("\n" + "=" * 70)
print("ALL EXPERIMENTS COMPLETE")
print("=" * 70)
print(f"Experiments run: {len(drift_configs)}")
print(f"Methods compared:")
print(f"  - ResNet-18 backbone: KL, JSD, Hotelling T², PSI, Energy Distance")
print(f"  - Raw pixel data:     VAE Reconstruction Error, VAE Latent KL")
print(f"  - Online detectors:   ADWIN (pixel mean + VAE error), DDM, EDDM")
print(f"New capabilities in this run:")
print(f"  - 4-way ResNet-KL classification: none / label / feature / concept")
print(f"  - Ground-truth timeline overlaid on every detection plot")
print(f"  - Per-method detection latency vs true drift onset")
print(f"  - Dedicated CONCEPT drift scenario (P(Y|X) change via label-flipping)")
print(f"Output saved to: ./output/")
